# DB7-017: BRB reliability fusion pilot

**Authorized scope: S1 and S15, seed42; 34 research neural fits.** One subject runs on each T4.

This tests whether eight-rule BRBs can estimate when each EMG-waveform (W), EMG-spectrum (S) and inertial (I) expert is reliable. The inertial expert retains ACC, gyroscope and magnetometer. SI and WSI are retrained comparison models.

## Fixed protocol

- Exercise B: E1 labels1–17; rest excluded; annotated repetition segments never cross windows.
- 200ms windows, 10ms stride; EMG12 + ACC36 + gyro36 + mag36.
- EMG20–450Hz bandpass and50Hz notch; inertial alignment per segment; no per-window centering.
- Train repetitions1/3/4/6; outer test2/5. Thirteen fixed epochs, Adam, batch512, dropout0.65; no augmentation or epoch selection.
- Per subject: twelve OOF neural fits (W/S/I × four held-out training repetitions), then five final fits (W/S/I/SI/WSI).
- OOF repetitions1/3/4 fit the meta components; OOF repetition6 calibrates scalar reliability. Cross-fitted temperatures construct reliability-training inputs.

The OOF networks share recordings between meta groups. Repetition6 is a development calibration holdout, not fully nested independent validation. Test2/5 has also been examined in earlier studies: all pilot test results are exploratory. No population confidence claim comes from overlapping windows or these two selected subjects.

## Read the code

Each implementation module is visible in its own cell. `%%writefile` stores that module; only the final cell starts training. No pip installation is required in the standard Kaggle GPU image.


In [ ]:
from pathlib import Path
import json

SOURCE = Path('/kaggle/working/db7_brb_src')
SOURCE.mkdir(parents=True, exist_ok=True)


### three_branch_model.py

Shared neural building blocks inherited unchanged from DB7-016.


In [ ]:
%%writefile /kaggle/working/db7_brb_src/three_branch_model.py
"""C1: frame-aligned EMG waveform, log-power and inertial encoders for DB7.

The caller supplies filtered, training-channel-standardized windows in this order:
EMG (12), ACC (36), gyroscope (36), magnetometer (36). This module does not read
recordings, fit the raw input scaler, choose data splits, or filter signals.

Before training, call ``model.fit_spectral_scaler(training_loader, split='train')``.
That loader must contain only already-standardized training windows. Spectral
statistics are registered buffers and travel with every model checkpoint.
"""

from __future__ import annotations

import io as checkpoint_io
from collections.abc import Iterable
from typing import Any

import torch
from torch import Tensor, nn
from torch.nn import functional as F


EXPECTED_PARAMETERS = 551_542
EXPECTED_PARAMETER_BREAKDOWN = {
    'waveform': 81_492,
    'spectral': 67_936,
    'inertial': 123_760,
    'fusion': 41_216,
    'temporal': 197_888,
    'attention': 4_161,
    'classifier': 35_089,
}
FRAME_SAMPLES = 200
FRAME_HOP = 100
WINDOW_SAMPLES = 400
N_FRAMES = 3


def unfold_frames(signal: Tensor) -> Tensor:
    """Return [batch, channels, 3, 200], using only the supplied 400 rows."""
    if signal.ndim != 3 or signal.shape[-1] != WINDOW_SAMPLES:
        raise ValueError(f'Expected [batch, channels, 400], received {tuple(signal.shape)}')
    return signal.unfold(-1, FRAME_SAMPLES, FRAME_HOP)


def log_power(z_emg: Tensor, hann: Tensor | None = None) -> Tensor:
    """Return unscaled log-power [batch, 12, 44, 3] for 20:10:450 Hz.

    ``z_emg`` has already received the fixed training input scaler. Explicit
    200-sample frames avoid the FFT-length-dependent framing of torch.stft.
    The FFT is at least float32, including inside an autocast context: CUDA
    half-precision FFTs cannot implement this non-power-of-two length.
    """
    if z_emg.ndim != 3 or z_emg.shape[1:] != (12, WINDOW_SAMPLES):
        raise ValueError(f'Expected [batch, 12, 400], received {tuple(z_emg.shape)}')
    if not z_emg.is_floating_point():
        raise TypeError('EMG input must be a floating-point tensor')
    fft_dtype = torch.float64 if z_emg.dtype == torch.float64 else torch.float32
    if hann is None:
        hann = torch.hann_window(FRAME_SAMPLES, periodic=True,
                                 device=z_emg.device, dtype=fft_dtype)
    else:
        hann = hann.to(device=z_emg.device, dtype=fft_dtype)
        if hann.shape != (FRAME_SAMPLES,):
            raise ValueError('Hann window must contain exactly 200 samples')
    with torch.autocast(device_type=z_emg.device.type, enabled=False):
        frames = unfold_frames(z_emg.to(dtype=fft_dtype))
        spectrum = torch.fft.rfft(frames * hann, n=FRAME_SAMPLES, dim=-1)
        power = spectrum.abs().square() / hann.square().sum()
        # [B, 12, frame, frequency] -> [B, 12, frequency, frame].
        return torch.log(power[..., 2:46] + 1e-8).permute(0, 1, 3, 2).contiguous()


def _conv_bn_relu(in_channels: int, out_channels: int, kernel: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv1d(in_channels, out_channels, kernel, padding=kernel // 2, bias=False),
        nn.BatchNorm1d(out_channels, eps=1e-5, momentum=0.1),
        nn.ReLU(),
    )


class ChannelSqueezeExcitation(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(channels, channels // 8), nn.ReLU(),
            nn.Linear(channels // 8, channels), nn.Sigmoid(),
        )

    def forward(self, x: Tensor) -> Tensor:
        return x * self.gate(x.mean(-1)).unsqueeze(-1)


class FrameMultiKernelBlock(nn.Module):
    """Three single-convolution paths, plus projected residual and frame SE."""
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.paths = nn.ModuleList(_conv_bn_relu(in_channels, 32, k) for k in (3, 5, 7))
        self.merge = nn.Sequential(
            nn.Conv1d(96, out_channels, 1, bias=False),
            nn.BatchNorm1d(out_channels, eps=1e-5, momentum=0.1),
        )
        self.skip = nn.Conv1d(in_channels, out_channels, 1, bias=False)
        self.se = ChannelSqueezeExcitation(out_channels)

    def forward(self, x: Tensor) -> Tensor:
        merged = self.merge(torch.cat([path(x) for path in self.paths], dim=1))
        return self.se(F.relu(merged + self.skip(x)))


class WaveformEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.stages = nn.Sequential(
            FrameMultiKernelBlock(12, 64), nn.MaxPool1d(2),
            FrameMultiKernelBlock(64, 96), nn.MaxPool1d(2),
        )
        self.projection = nn.Linear(192, 96)

    def forward(self, frames: Tensor) -> Tensor:
        # The same encoder processes every frame; batch and frame remain distinct.
        batch = frames.shape[0]
        x = frames.permute(0, 2, 1, 3).reshape(batch * N_FRAMES, 12, FRAME_SAMPLES)
        x = self.stages(x)
        x = F.relu(self.projection(torch.cat([x.mean(-1), x.amax(-1)], dim=1)))
        return x.reshape(batch, N_FRAMES, 96).transpose(1, 2).contiguous()


class SpectralEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        for incoming, outgoing, kernel, stride in ((12, 32, 5, 2),
                                                   (32, 64, 5, 2),
                                                   (64, 96, 3, 1)):
            layers.extend([
                nn.Conv2d(incoming, outgoing, (kernel, 1), stride=(stride, 1),
                          padding=(kernel // 2, 0), bias=False),
                nn.BatchNorm2d(outgoing, eps=1e-5, momentum=0.1), nn.ReLU(),
            ])
        self.stages = nn.Sequential(*layers)
        self.projection = nn.Conv1d(384, 96, 1, bias=True)

    def forward(self, x: Tensor) -> Tensor:
        x = self.stages(x)
        # Preserve four ordered feature-frequency regions, rather than collapsing
        # absolute frequency location into a single global mean/max vector.
        regions = torch.stack([x[:, :, begin:end, :].mean(2)
                               for begin, end in ((0, 3), (3, 6), (6, 9), (9, 11))], dim=2)
        # Channel-major: each channel retains regions low -> high in adjacent slots.
        return F.relu(self.projection(regions.flatten(1, 2)))


class InertialModalityEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.stages = nn.Sequential(
            _conv_bn_relu(36, 48, 5), nn.AvgPool1d(4),
            _conv_bn_relu(48, 64, 5),
        )
        self.projection = nn.Linear(128, 48)

    def forward(self, frames: Tensor) -> Tensor:
        x = self.stages(frames)
        return F.relu(self.projection(torch.cat([x.mean(-1), x.amax(-1)], dim=1)))


class InertialEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.modalities = nn.ModuleList(InertialModalityEncoder() for _ in range(3))
        self.dynamic_projection = nn.Linear(144, 128)
        self.mean_projection = nn.Linear(108, 128)

    def forward(self, frames: Tensor) -> Tensor:
        batch = frames.shape[0]
        x = frames.permute(0, 2, 1, 3).reshape(batch * N_FRAMES, 108, FRAME_SAMPLES)
        encoded = [encoder(x[:, i * 36:(i + 1) * 36, :])
                   for i, encoder in enumerate(self.modalities)]
        dynamic = self.dynamic_projection(torch.cat(encoded, dim=1))
        static = self.mean_projection(x.mean(-1))
        # The mean is an additional feature; it is never subtracted from frames.
        output = F.relu(dynamic + static)
        return output.reshape(batch, N_FRAMES, 128).transpose(1, 2).contiguous()


class ResidualTemporalBlock(nn.Module):
    def __init__(self, dilation: int, dropout: float) -> None:
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv1d(128, 192, 3, dilation=dilation, padding=dilation, bias=False),
            nn.BatchNorm1d(192, eps=1e-5, momentum=0.1), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(192, 128, 1, bias=False),
            nn.BatchNorm1d(128, eps=1e-5, momentum=0.1), nn.Dropout(dropout),
        )

    def forward(self, x: Tensor) -> Tensor:
        return F.relu(x + self.layers(x))


class ThreeBranchC1(nn.Module):
    """Exact 551,542-parameter C1, compatible with Trainer's model(x) call."""
    def __init__(self, dropout: float = 0.15) -> None:
        super().__init__()
        self.waveform = WaveformEncoder()
        self.spectral = SpectralEncoder()
        self.inertial = InertialEncoder()
        self.fusion = nn.Sequential(
            nn.Conv1d(320, 128, 1, bias=False),
            nn.BatchNorm1d(128, eps=1e-5, momentum=0.1), nn.ReLU(),
        )
        self.temporal = nn.Sequential(ResidualTemporalBlock(1, dropout),
                                      ResidualTemporalBlock(2, dropout))
        self.attention = nn.Sequential(nn.Linear(128, 32), nn.Tanh(), nn.Linear(32, 1))
        self.classifier = nn.Sequential(nn.Linear(256, 128), nn.ReLU(),
                                        nn.Dropout(dropout), nn.Linear(128, 17))
        self.register_buffer('hann', torch.hann_window(FRAME_SAMPLES, periodic=True))
        self.register_buffer('spectral_mean', torch.zeros(12, 44))
        self.register_buffer('spectral_std', torch.ones(12, 44))
        self.register_buffer('spectral_constant', torch.zeros(12, 44, dtype=torch.bool))
        self.register_buffer('spectral_fitted', torch.tensor(False))
        self.register_buffer('spectral_fit_frames', torch.tensor(0, dtype=torch.long))
        if self.count_params() != EXPECTED_PARAMETERS:
            raise AssertionError(f'C1 parameter mismatch: {self.count_params()}')

    def count_params(self) -> int:
        return sum(parameter.numel() for parameter in self.parameters())

    def parameter_breakdown(self) -> dict[str, int]:
        return {name: sum(parameter.numel() for parameter in getattr(self, name).parameters())
                for name in EXPECTED_PARAMETER_BREAKDOWN}

    @torch.no_grad()
    def fit_spectral_scaler(self, training_batches: Iterable[Any], *, split: str = 'train',
                            std_floor: float = 1e-6) -> dict[str, Any]:
        """Fit frozen moments without running a CNN or updating BatchNorm.

        The iterable yields standardized x or (x,y) batches. One observation for
        each channel/frequency bin is one frame; all three frames of every
        training window receive equal weight. Overlap is intentional. Population
        variance uses a stable float64 batched merge; SD below ``std_floor`` is
        flagged and clamped. A second fit is rejected to avoid accidental reuse
        on validation/test; create a fresh model for a new subject or fold.

        The explicit split guard cannot establish arbitrary generator provenance.
        The caller must save source/window hashes. A DataLoader exposing a dataset
        with a ``meta['split']`` column receives an additional provenance check.
        """
        if split != 'train':
            raise ValueError('Spectral scaler may be fitted only on the train split')
        if bool(self.spectral_fitted.item()):
            raise RuntimeError('Spectral scaler is already fitted; use a fresh model for another fold')
        if std_floor <= 0:
            raise ValueError('std_floor must be positive')
        dataset = getattr(training_batches, 'dataset', None)
        metadata = getattr(dataset, 'meta', None)
        if metadata is not None and 'split' in metadata:
            if set(metadata['split'].unique()) != {'train'}:
                raise ValueError('Spectral fitting DataLoader contains non-training metadata')
        device = self.spectral_mean.device
        count = 0
        mean = torch.zeros(12, 44, dtype=torch.float64, device=device)
        m2 = torch.zeros_like(mean)
        for batch in training_batches:
            x = batch[0] if isinstance(batch, (tuple, list)) else batch
            x = torch.as_tensor(x, device=device, dtype=self.spectral_mean.dtype)
            if x.ndim != 3 or x.shape[1:] != (120, WINDOW_SAMPLES) or x.shape[0] == 0:
                raise ValueError('Spectral scaler requires nonempty [B,120,400] training batches')
            if not bool(torch.isfinite(x).all().item()):
                raise ValueError('Nonfinite training input in spectral scaler fit')
            values = log_power(x[:, :12], self.hann).to(torch.float64)
            batch_count = values.shape[0] * N_FRAMES
            batch_mean = values.mean(dim=(0, 3))
            batch_m2 = (values - batch_mean[None, :, :, None]).square().sum(dim=(0, 3))
            combined_count = count + batch_count
            delta = batch_mean - mean
            m2 += batch_m2 + delta.square() * (count * batch_count / combined_count)
            mean += delta * (batch_count / combined_count)
            count = combined_count
        if not count:
            raise ValueError('Cannot fit the spectral scaler on an empty iterable')
        std = (m2 / count).clamp_min(0).sqrt()
        self.spectral_mean.copy_(mean)
        self.spectral_constant.copy_(std < std_floor)
        self.spectral_std.copy_(std.clamp_min(std_floor))
        self.spectral_fit_frames.fill_(count)
        self.spectral_fitted.fill_(True)
        return {
            'split': 'train', 'windows': count // N_FRAMES, 'frames': count,
            'statistic': 'float64_population_moments_over_training_windows_and_three_frames',
            'std_floor': float(std_floor),
            'constant_bins': int(self.spectral_constant.sum().item()),
            'mean': self.spectral_mean.detach().cpu().tolist(),
            'std': self.spectral_std.detach().cpu().tolist(),
            'constant': self.spectral_constant.detach().cpu().tolist(),
        }

    def forward_features(self, x: Tensor, *, retain_sequences: bool = False) -> dict[str, Any]:
        if x.ndim != 3 or x.shape[1:] != (120, WINDOW_SAMPLES):
            raise ValueError(f'C1 expects [B,120,400], received {tuple(x.shape)}')
        if not bool(self.spectral_fitted.item()):
            raise RuntimeError('Fit the spectral scaler using training windows before model(x)')
        frames = unfold_frames(x)
        waveform = self.waveform(frames[:, :12])
        power = log_power(x[:, :12], self.hann)
        scaled_power = ((power - self.spectral_mean[None, :, :, None]) /
                        self.spectral_std[None, :, :, None])
        spectral = self.spectral(scaled_power)
        inertial = self.inertial(frames[:, 12:])
        fused = self.temporal(self.fusion(torch.cat([waveform, spectral, inertial], dim=1)))
        scores = self.attention(fused.transpose(1, 2)).squeeze(-1)
        attention = torch.softmax(scores, dim=-1)
        weighted = (fused * attention[:, None, :]).sum(-1)
        embedding = torch.cat([weighted, fused.mean(-1)], dim=1)
        result = {
            'logits': self.classifier(embedding), 'embedding': embedding,
            'branch_embeddings': {'waveform': waveform.mean(-1),
                                  'spectral': spectral.mean(-1),
                                  'inertial': inertial.mean(-1)},
            'attention': attention,
        }
        if retain_sequences:
            result['sequences'] = {'waveform': waveform, 'spectral': spectral,
                                   'inertial': inertial, 'fused': fused}
        return result

    def forward(self, x: Tensor) -> Tensor:
        return self.forward_features(x)['logits']


def model_preflight(device: str | torch.device = 'cpu') -> dict[str, Any]:
    """Meaningful synthetic checks for the installed Kaggle PyTorch runtime.

    Does not touch recordings or disk, install packages, or start an experiment.
    CPU and applicable CUDA RNG state are restored before return.
    """
    target = torch.device(device)
    # torch.manual_seed also seeds CUDA generators. Preserve every initialized
    # device's stream instead of changing an unused second Kaggle GPU's RNG.
    cuda_devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
    with torch.random.fork_rng(devices=cuda_devices):
        torch.manual_seed(941)
        model = ThreeBranchC1().to(target)
        assert model.parameter_breakdown() == EXPECTED_PARAMETER_BREAKDOWN
        x = torch.randn(3, 120, WINDOW_SAMPLES, device=target)
        try:
            model(x)
        except RuntimeError as error:
            assert 'spectral scaler' in str(error)
        else:
            raise AssertionError('Unfitted spectral scaling must block model inference')
        try:
            model.fit_spectral_scaler([x], split='test')
        except ValueError as error:
            assert 'train split' in str(error)
        else:
            raise AssertionError('The spectral scaler must reject non-training splits')
        frames = unfold_frames(x)
        assert frames.shape == (3, 120, 3, 200)
        for frame in range(N_FRAMES):
            torch.testing.assert_close(frames[:, :, frame], x[:, :, frame * 100:frame * 100 + 200])
        # Independent explicit-frame transform verifies frequency/frame ordering.
        hann = torch.hann_window(200, periodic=True, device=target)
        reference = torch.stack([
            torch.log(torch.fft.rfft(x[:, :12, begin:begin + 200] * hann, dim=-1)
                      .abs().square()[..., 2:46] / hann.square().sum() + 1e-8)
            for begin in range(0, 201, 100)], dim=-1)
        torch.testing.assert_close(log_power(x[:, :12]), reference)
        before_bn = {name: value.clone() for name, value in model.named_buffers()
                     if 'running_' in name or 'num_batches_tracked' in name}
        statistics = model.fit_spectral_scaler([(x[:2], torch.zeros(2)),
                                                (x[2:], torch.zeros(1))], split='train')
        assert statistics['windows'] == 3 and statistics['frames'] == 9
        try:
            model.fit_spectral_scaler([x], split='train')
        except RuntimeError as error:
            assert 'already fitted' in str(error)
        else:
            raise AssertionError('A fitted spectral scaler must reject accidental refitting')
        reference_double = reference.double()
        expected_mean = reference_double.mean(dim=(0, 3))
        expected_std = reference_double.permute(1, 2, 0, 3).reshape(12, 44, -1).std(-1, correction=0)
        torch.testing.assert_close(model.spectral_mean, expected_mean.float(), rtol=2e-5, atol=2e-6)
        torch.testing.assert_close(model.spectral_std, expected_std.float(), rtol=2e-5, atol=2e-6)
        for name, value in model.named_buffers():
            if name in before_bn:
                torch.testing.assert_close(value, before_bn[name], rtol=0, atol=0)
        frozen_scaler = {name: value.clone() for name, value in model.named_buffers()
                         if name.startswith('spectral_')}
        model.train()
        outputs = model.forward_features(x, retain_sequences=True)
        assert outputs['logits'].shape == (3, 17)
        assert outputs['embedding'].shape == (3, 256)
        assert outputs['attention'].shape == (3, 3)
        for branch, width in (('waveform', 96), ('spectral', 96), ('inertial', 128), ('fused', 128)):
            assert outputs['sequences'][branch].shape == (3, width, 3)
        torch.testing.assert_close(outputs['attention'].sum(-1), torch.ones(3, device=target))
        loss = F.cross_entropy(outputs['logits'], torch.tensor([0, 8, 16], device=target))
        loss.backward()
        for name, module in (('waveform', model.waveform), ('spectral', model.spectral),
                             ('acc', model.inertial.modalities[0]),
                             ('gyro', model.inertial.modalities[1]),
                             ('mag', model.inertial.modalities[2]),
                             ('inertial_mean', model.inertial.mean_projection),
                             ('fusion', model.fusion)):
            gradients = [p.grad for p in module.parameters() if p.grad is not None]
            assert gradients and all(bool(torch.isfinite(g).all().item()) for g in gradients), name
            assert any(bool(g.abs().sum().item() > 0) for g in gradients), name
        for name, value in model.named_buffers():
            if name in frozen_scaler:
                torch.testing.assert_close(value, frozen_scaler[name], rtol=0, atol=0)
        model.eval()
        with torch.no_grad():
            expected = model(x)
            # An input to another branch cannot change waveform/spectral tokens.
            changed = x.clone()
            changed[:, 12:] += 1.75
            original_features = model.forward_features(x, retain_sequences=True)
            altered_features = model.forward_features(changed, retain_sequences=True)
            for name in ('waveform', 'spectral'):
                torch.testing.assert_close(original_features['sequences'][name],
                                           altered_features['sequences'][name], rtol=0, atol=0)
            checkpoint = checkpoint_io.BytesIO()
            torch.save(model.state_dict(), checkpoint)
            checkpoint.seek(0)
            restored = ThreeBranchC1().to(target)
            restored.load_state_dict(torch.load(checkpoint, map_location=target, weights_only=True))
            restored.eval()
            torch.testing.assert_close(restored(x), expected, rtol=0, atol=0)
        return {'success': True, 'parameters': model.count_params(),
                'parameter_breakdown': model.parameter_breakdown(), 'device': str(target),
                'torch_version': torch.__version__, 'frames': 3, 'frequency_bins': 44,
                'output_shape': list(expected.shape),
                'checks': ['exact_parameter_count', 'frame_alignment', 'explicit_fft_reference',
                           'unfitted_nontraining_and_refit_guards',
                           'training_spectral_moments', 'scaler_fit_does_not_update_batchnorm',
                           'frozen_scaler', 'forward_backward_all_branches',
                           'branch_input_isolation', 'checkpoint_round_trip']}


if __name__ == '__main__':
    import json
    print(json.dumps(model_preflight('cuda' if torch.cuda.is_available() else 'cpu'), indent=2))


### ablation_model.py

W, S and I experts plus independently trained SI and WSI controls.


In [ ]:
%%writefile /kaggle/working/db7_brb_src/ablation_model.py
"""Retrained branch subsets. Full WSI is numerically identical to original C1."""
import torch
from torch import nn
from three_branch_model import ThreeBranchC1, unfold_frames, log_power

ARMS=('W','S','I','WS','WI','SI','WSI')
NAMES={'W':'waveform','S':'spectral','I':'inertial'}
SLICES={'W':(0,96),'S':(96,192),'I':(192,320)}

class AblationC1(ThreeBranchC1):
    def __init__(self,arm,dropout=.15):
        assert arm in ARMS
        super().__init__(dropout)
        self.arm=arm
        self.active_names=[NAMES[k] for k in arm]
        indices=[i for k in arm for i in range(*SLICES[k])]
        if arm!='WSI':
            old=self.fusion[0]
            # Preserve the shared initialization; restore RNG after constructor.
            with torch.random.fork_rng():
                replacement=nn.Conv1d(len(indices),128,1,bias=False)
            with torch.no_grad():replacement.weight.copy_(old.weight[:,indices])
            self.fusion[0]=replacement
            for k,name in NAMES.items():
                if k not in arm:delattr(self,name)

    def forward_features(self,x,retain_sequences=False):
        frames=unfold_frames(x);seq={}
        if 'W' in self.arm:seq['waveform']=self.waveform(frames[:,:12])
        if 'S' in self.arm:
            if not self.spectral_fitted:raise RuntimeError('Fit spectral scaler first')
            p=log_power(x[:,:12],self.hann)
            seq['spectral']=self.spectral((p-self.spectral_mean[None,:,:,None])/self.spectral_std[None,:,:,None])
        if 'I' in self.arm:seq['inertial']=self.inertial(frames[:,12:])
        fused=self.temporal(self.fusion(torch.cat(list(seq.values()),dim=1)))
        attention=self.attention(fused.transpose(1,2)).squeeze(-1).softmax(-1)
        embedding=torch.cat([(fused*attention[:,None]).sum(-1),fused.mean(-1)],dim=1)
        out=dict(logits=self.classifier(embedding),embedding=embedding,
                 attention=attention,branch_embeddings={k:v.mean(-1) for k,v in seq.items()})
        if retain_sequences:out['sequences']={**seq,'fused':fused}
        return out

def preflight():
    """Exact full-model parity; gradient coverage and excluded-input isolation."""
    x=torch.randn(3,120,400,device='cuda')
    torch.manual_seed(77);ref=ThreeBranchC1().cuda()
    ref.fit_spectral_scaler([x]);ref.eval()
    torch.manual_seed(77);full=AblationC1('WSI').cuda()
    full.load_state_dict(ref.state_dict());full.eval()
    torch.testing.assert_close(full(x),ref(x),rtol=0,atol=0)
    rows=[]
    for arm in ARMS:
        model=AblationC1(arm).cuda()
        if 'S' in arm:model.fit_spectral_scaler([x])
        model.train();out=model(x)
        nn.functional.cross_entropy(out,torch.tensor([0,8,16],device='cuda')).backward()
        assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
        model.eval()
        changed=x.clone()
        if 'I' not in arm:changed[:,12:]+=10
        if arm=='I':changed[:,:12]+=10
        torch.testing.assert_close(model(x),model(changed),rtol=0,atol=0)
        rows.append(dict(arm=arm,parameters=model.count_params()))
    return rows


### tc_support.py

Original segment filtering, inertial alignment and training-only normalization helpers.


In [ ]:
%%writefile /kaggle/working/db7_brb_src/tc_support.py
"""C1 under TC-AiFusion's split and training budget, retaining all inertial sensors.

No TC feature images or external-window history are used: these would replace C1.
All preprocessing fits use training repetitions only. Test is evaluated once,
after the fixed final epoch. This is annotation-assisted offline classification.
"""
import gc, hashlib, json, random, time, traceback, zipfile
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt, resample_poly
import torch
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from three_branch_model import ThreeBranchC1, model_preflight


class Settings:
    SUBJECTS = list(range(1, 21))
    TRAIN_REPS = [1, 3, 4, 6]
    TEST_REPS = [2, 5]
    FS = 2000
    WINDOW = 400  # 200 ms
    STEP = 20     # 10 ms
    BATCH_SIZE = 512
    EPOCHS = 13
    DROPOUT = 0.65  # Match TC-AiFusion's training regularization setting.
    SEED = 42
    SMOKE = False
    INPUT = None
    OUTPUT = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()
    AUTOMATION = {}


def write_json(path, value):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(value,indent=2),encoding='utf-8')


def find_input():
    if Settings.INPUT is not None:return Path(Settings.INPUT)
    for candidate in sorted(Path('/kaggle/input').rglob('Subject_1')):
        if candidate.is_dir():return candidate.parent
    raise FileNotFoundError('Attach rayaanraza1/ninapro-db7 or set Settings.INPUT to the subject-folder root.')


def aligned_interval(array, emg_length, start, end):
    """TC alignment rule, independently applied to ACC, gyro and magnetometer."""
    if len(array)==emg_length:return array[start:end].astype(np.float32,copy=True)
    ratio=len(array)/float(emg_length)
    a0=max(0,min(int(np.floor(start*ratio)),len(array)-1))
    a1=max(a0+1,min(int(np.ceil(end*ratio)),len(array)))
    segment=array[a0:a1];length=end-start
    divisor=np.gcd(len(segment),length)
    result=resample_poly(segment,length//divisor,len(segment)//divisor,axis=0).astype(np.float32)
    if len(result)<length:result=np.vstack([result,np.repeat(result[-1:],length-len(result),axis=0)])
    return result[:length]


def load_subject(root, subject, output):
    directory=root/f'Subject_{subject}'
    if not directory.exists():directory=root/f'S{subject}'
    files=sorted(directory.rglob('*_E1_*.mat'))
    if len(files)!=1:raise ValueError(f'S{subject}: expected one E1 file, found {len(files)}')
    data=loadmat(files[0],variable_names=['emg','acc','gyro','mag','restimulus','rerepetition','subject','exercise'])
    def sensor(key,channels):
        x=np.asarray(data[key],dtype=np.float32)
        if x.ndim!=2:raise ValueError(f'{key}: not 2D')
        if x.shape[1]!=channels and x.shape[0]==channels:x=x.T
        if x.shape[1]!=channels or not np.isfinite(x).all():raise ValueError(f'{key}: invalid channels/values')
        return x
    emg=sensor('emg',12)
    inertial=[sensor(key,36) for key in ['acc','gyro','mag']]
    labels=np.asarray(data['restimulus']).reshape(-1).astype(int)
    reps=np.asarray(data['rerepetition']).reshape(-1).astype(int)
    assert len(emg)==len(labels)==len(reps), 'Reject silent label/signal truncation'
    assert set(np.unique(labels))==set(range(18))
    edges=np.r_[0,np.flatnonzero(np.diff(labels))+1,len(labels)]
    sos=butter(4,[20,450],btype='bandpass',fs=Settings.FS,output='sos')
    b,a=iirnotch(50,30,fs=Settings.FS)
    segments=[];records=[]
    for start,end in zip(edges[:-1],edges[1:]):
        gesture=int(labels[start])
        if gesture==0:continue
        native=np.unique(reps[start:end])
        if len(native)!=1 or native[0] not in range(1,7):
            raise ValueError('Gesture run must contain exactly one valid repetition; do not assign by majority')
        repetition=int(native[0]);split='train' if repetition in Settings.TRAIN_REPS else 'test'
        filtered=sosfiltfilt(sos,emg[start:end],axis=0)
        filtered=filtfilt(b,a,filtered,axis=0).astype(np.float32)
        x=np.concatenate([filtered]+[aligned_interval(v,len(emg),int(start),int(end)) for v in inertial],axis=1)
        if not np.isfinite(x).all():raise ValueError('Nonfinite filtered/aligned signal')
        segments.append(x)
        records.append(dict(subject=subject,gesture=gesture,native_repetition=repetition,split=split,
            run_start=int(start),run_end=int(end),file_name=files[0].name,segment=len(segments)-1))
    inventory=pd.DataFrame(records)
    for gesture in range(1,18):
        rows=inventory[inventory.gesture==gesture]
        assert len(rows)==6 and set(rows.native_repetition)==set(range(1,7))
    # TC normalization: every sample of each active training segment counted once.
    total=np.zeros(120,np.float64);squares=total.copy();count=0
    for rec in records:
        if rec['split']=='train':
            x=segments[rec['segment']].astype(np.float64)
            total+=x.sum(0);squares+=np.square(x).sum(0);count+=len(x)
    mean=total/count;variance=np.maximum(squares/count-mean*mean,1e-12)
    std=np.sqrt(variance)
    np.savez_compressed(output/'input_scaler.npz',mean=mean.astype(np.float32),std=std.astype(np.float32),training_sample_count=count)
    inventory.to_csv(output/'repetition_inventory.csv',index=False)
    write_json(output/'identity.json',dict(folder_subject=subject,internal_subject=int(np.asarray(data.get('subject',subject)).item()),
        file=files[0].name,emg_sha256=hashlib.sha256(emg.tobytes()).hexdigest(),
        sensor_shapes={key:list(value.shape) for key,value in zip(['emg','acc','gyro','mag'],[emg]+inertial)},
        alignment_fallback={key:len(value)!=len(emg) for key,value in zip(['acc','gyro','mag'],inertial)}))
    return segments,inventory,mean.astype(np.float32),std.astype(np.float32)


class TCWindows(Dataset):
    def __init__(self,segments,inventory,mean,std,split):
        self.segments=segments;self.mean=mean;self.std=std
        rows=[]
        for rec in inventory[inventory.split==split].to_dict('records'):
            for offset in range(0,rec['run_end']-rec['run_start']-Settings.WINDOW+1,Settings.STEP):
                start=rec['run_start']+offset;end=start+Settings.WINDOW
                phase=((start+end)/2-rec['run_start'])/(rec['run_end']-rec['run_start'])
                rows.append(dict(**rec,offset=offset,window_start=start,window_end=end,
                    phase_fraction=phase,phase='early' if phase<1/3 else 'middle' if phase<2/3 else 'late'))
        self.meta=pd.DataFrame(rows)
        self.y=self.meta.gesture.to_numpy()-1
        self.locations=self.meta[['segment','offset']].to_numpy()
        assert not self.meta.duplicated(['subject','gesture','native_repetition','window_start']).any()
    def __len__(self):return len(self.y)
    def __getitem__(self,index):
        seg,start=self.locations[index]
        x=self.segments[seg][start:start+Settings.WINDOW]
        return torch.from_numpy(((x-self.mean)/self.std).T.copy()),int(self.y[index])


def set_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)


def evaluate(model,dataset,device,path):
    path.mkdir(exist_ok=True)
    model.eval();probs=[];embeddings=[];attention=[]
    branch={key:[] for key in model.active_names}
    with torch.no_grad():
        for x,y in DataLoader(dataset,batch_size=Settings.BATCH_SIZE,shuffle=False):
            out=model.forward_features(x.to(device))
            probs.append(out['logits'].softmax(1).cpu().numpy());embeddings.append(out['embedding'].cpu().numpy())
            attention.append(out['attention'].cpu().numpy())
            for key in branch:branch[key].append(out['branch_embeddings'][key].cpu().numpy())
    p=np.concatenate(probs);pred=p.argmax(1)
    assert np.isfinite(p).all() and np.allclose(p.sum(1),1,atol=1e-5)
    frame=dataset.meta.copy();frame['seed_base']=Settings.SEED;frame['arm']=model.arm;frame['prediction']=pred+1;frame['correct']=pred==dataset.y;frame['confidence']=p.max(1)
    if path.name == 'test':
        frame.to_csv(path/'predictions.csv',index=False)
        arrays=dict(y_true=dataset.y, probabilities=p)
        if Settings.SEED==42:
            arrays.update(embeddings=np.concatenate(embeddings),attention=np.concatenate(attention),
                          **{k:np.concatenate(v) for k,v in branch.items()})
        np.savez_compressed(path/'probabilities_embeddings.npz',**arrays)
    errors=frame.groupby(['subject','gesture','native_repetition']).agg(windows=('correct','size'),correct=('correct','sum')).reset_index()
    errors['wrong']=errors.windows-errors.correct;errors['error_percent']=100*errors.wrong/errors.windows
    errors.to_csv(path/'gesture_errors.csv',index=False)
    phase=frame.groupby(['gesture','native_repetition','phase']).agg(windows=('correct','size'),correct=('correct','sum')).reset_index()
    phase['wrong']=phase.windows-phase.correct;phase.to_csv(path/'phase_errors.csv',index=False)
    confusion=pd.crosstab(frame.gesture,frame.prediction).reindex(index=range(1,18),columns=range(1,18),fill_value=0)
    confusion.to_csv(path/'confusion.csv')
    tp=np.diag(confusion);den=confusion.sum(axis=0).to_numpy()+confusion.sum(axis=1).to_numpy()
    f1=2*tp/np.maximum(den,1)
    metrics=dict(accuracy=float(frame.correct.mean()),macro_f1=float(f1.mean()),n_windows=len(frame),
        wrong=int((~frame.correct).sum()),nll=float(-np.log(p[np.arange(len(p)),dataset.y].clip(1e-12)).mean()))
    write_json(path/'metrics.json',metrics)
    return metrics




### brb_meta.py

Temperature calibration, reliability indicators, analytical BRB/RIMER and comparison gates.


In [ ]:
%%writefile /kaggle/working/db7_brb_src/brb_meta.py
"""Training-only reliability fusion for DB7 W/S/I expert logits.

Labels are zero based. fit_meta accepts OOF predictions for repetitions 1/3/4/6
only. Repetition 6 is reserved for reliability calibration, not model fitting,
temperature selection or global fusion weights. The caller must supply shifts
computed against each OOF base model's own training-only signal references.

This is nested development, NOT independent meta cross-validation: OOF base
models may share training recordings. Outer test predictions never enter fit.
"""
from __future__ import annotations

import csv
import hashlib
import json
from pathlib import Path

import numpy as np
from scipy.optimize import minimize
from scipy.special import expit, logit, logsumexp, softmax

VERSION = "db7-brb-meta-v1"
EXPERTS = ("W", "S", "I")
FIT_REPS = (1, 3, 4)
CAL_REP = 6
EPS = 1e-9
HEAD_L2 = 0.01
CAL_L2 = 0.01
ALPHA_L2 = 0.005
MIN_EVENTS = 5
MAXITER = 180
RULE_BITS = np.array([[int(x) for x in f"{r:03b}"] for r in range(8)])


def _inputs(logits, shifts, y=None, repetitions=None):
    z = np.asarray(logits, dtype=np.float64)
    s = np.asarray(shifts, dtype=np.float64)
    if z.ndim != 3 or z.shape[1:] != (3, 17) or len(z) == 0:
        raise ValueError("logits must have nonempty shape [N,3,17]")
    if s.shape != z.shape[:2] or not np.isfinite(z).all() or not np.isfinite(s).all():
        raise ValueError("finite shifts [N,3] and logits required")
    if np.any((s < 0) | (s > 1)):
        raise ValueError("training-reference shifts must be in [0,1]")
    if y is None:
        return z, s
    yy = np.asarray(y)
    rr = np.asarray(repetitions)
    if yy.shape != (len(z),) or rr.shape != yy.shape:
        raise ValueError("y and repetitions must have shape [N]")
    if not np.isin(yy, np.arange(17)).all():
        raise ValueError("y must contain zero-based labels 0..16")
    if not np.isin(rr, (*FIT_REPS, CAL_REP)).all():
        raise ValueError("fit_meta accepts only training repetitions 1,3,4,6; test forbidden")
    if set(rr.tolist()) != {1, 3, 4, 6}:
        raise ValueError("all meta-fit repetitions 1/3/4 and calibration repetition 6 required")
    return z, s, yy.astype(np.int64), rr.astype(np.int64)


def _group_weights(groups):
    """Equal repetition weight; do not treat differing window counts as trials."""
    groups = np.asarray(groups)
    levels, counts = np.unique(groups, return_counts=True)
    return np.array([1.0 / (len(levels) * counts[np.searchsorted(levels, g)]) for g in groups])


def _binary_loss(target, predicted, weights):
    p = np.clip(predicted, EPS, 1 - EPS)
    return float(-np.sum(weights * (target * np.log(p) + (1 - target) * np.log1p(-p))))


def _optimization(result):
    return {"success": bool(result.success), "message": str(result.message),
            "iterations": int(result.nit), "objective": float(result.fun)}


def fit_temperatures(logits, y, groups):
    """Positive scalar temperature per expert, using supplied development rows."""
    weights = _group_weights(groups)
    temperatures, diagnostics = [], []
    for j in range(3):
        z = logits[:, j]
        def objective(theta):
            zz = z / np.exp(theta[0])
            p = softmax(zz, axis=1)
            loss = np.sum(weights * (logsumexp(zz, axis=1) - zz[np.arange(len(y)), y]))
            grad = np.sum(weights * (zz[np.arange(len(y)), y] - np.sum(p * zz, axis=1)))
            return float(loss + 0.001 * theta[0] ** 2), np.array([grad + .002 * theta[0]])
        opt = minimize(objective, [0.], jac=True, method="L-BFGS-B",
                       bounds=[(np.log(.05), np.log(20.))], options={"maxiter": MAXITER})
        if not np.isfinite(opt.fun):
            raise RuntimeError("nonfinite temperature optimization")
        temperatures.append(float(np.exp(opt.x[0])))
        diagnostics.append(_optimization(opt))
    return temperatures, diagnostics


def _probabilities(logits, temperatures):
    return softmax(logits / np.asarray(temperatures)[None, :, None], axis=2)


def indicators(probabilities, shifts):
    """Expert entropy, mean pairwise TV and precomputed training deviation."""
    p = np.asarray(probabilities, dtype=float)
    entropy = -np.sum(p * np.log(np.clip(p, EPS, 1)), axis=2) / np.log(17.)
    disagreement = np.zeros(p.shape[:2])
    for j in range(3):
        disagreement[:, j] = sum(.5 * np.abs(p[:, j] - p[:, k]).sum(1)
                                for k in range(3) if k != j) / 2
    return np.clip(np.stack([entropy, disagreement, shifts], axis=-1), 0, 1)


def rule_activations(q):
    """Product reference matching: [...,3] -> [...,8], sum exactly one."""
    q = np.asarray(q, dtype=float)
    if q.shape[-1] != 3 or not np.isfinite(q).all() or np.any((q < 0) | (q > 1)):
        raise ValueError("rule indicators must be finite [...,3] in [0,1]")
    a = np.prod(np.where(RULE_BITS, q[..., None, :], 1 - q[..., None, :]), axis=-1)
    return a / a.sum(axis=-1, keepdims=True)


def rimer_correct(activations, correct_beliefs, return_jacobian=False):
    """Analytical ER/RIMER for complete binary rule conclusions.

    A_n=prod(1-w+w*beta_n), B=prod(1-w), beta_n=(A_n-B)/(A0+A1-2B).
    Normalized activation is rule evidence weight. Returned jacobian is with
    respect to each rule's correctness belief (NOT its logit).
    """
    w = np.asarray(activations, dtype=float)
    b = np.asarray(correct_beliefs, dtype=float)
    if w.shape[-1] != 8 or b.shape != (8,):
        raise ValueError("eight activations and eight correctness beliefs required")
    if not np.isfinite(w).all() or not np.isfinite(b).all() or np.any((b < 0) | (b > 1)):
        raise ValueError("invalid ER beliefs")
    if np.any((w < 0) | (w > 1)) or not np.allclose(w.sum(-1), 1):
        raise ValueError("ER activations must be normalized")
    f1 = 1 - w + w * b
    f0 = 1 - w + w * (1 - b)
    a1, a0, bb = f1.prod(-1), f0.prod(-1), (1 - w).prod(-1)
    denom = a1 + a0 - 2 * bb
    if np.any(denom <= 0):
        raise FloatingPointError("degenerate ER normalization")
    p = np.clip((a1 - bb) / denom, 0, 1)
    if not return_jacobian:
        return p
    # Product excluding each factor handles exact zero factors at rule vertices.
    da1 = np.stack([w[..., k] * np.delete(f1, k, axis=-1).prod(-1) for k in range(8)], -1)
    da0 = -np.stack([w[..., k] * np.delete(f0, k, axis=-1).prod(-1) for k in range(8)], -1)
    jac = (da1 * denom[..., None] - (a1 - bb)[..., None] * (da1 + da0)) / denom[..., None] ** 2
    return p, jac


def _fit_alpha(p, y, weights):
    true_p = p[np.arange(len(p))[:, None], np.arange(3)[None, :], y[:, None]]
    def objective(theta):
        alpha = softmax(theta)
        mixture = np.clip(true_p @ alpha, EPS, 1)
        loss = -np.sum(weights * np.log(mixture)) + ALPHA_L2 * np.sum(theta ** 2)
        da = -np.sum(weights[:, None] * true_p / mixture[:, None], axis=0)
        grad = alpha * (da - alpha @ da) + 2 * ALPHA_L2 * theta
        return float(loss), grad
    opt = minimize(objective, np.zeros(3), jac=True, method="L-BFGS-B",
                   bounds=[(-6, 6)] * 3, options={"maxiter": MAXITER})
    return softmax(opt.x).tolist(), _optimization(opt)


def _raw_head(head, q):
    if head["kind"] == "constant":
        return np.full(len(q), head["value"])
    if head["kind"] == "logistic":
        return expit(np.column_stack([np.ones(len(q)), q - .5]) @ np.asarray(head["parameters"]))
    qq = q.copy()
    if head["kind"] == "brb_no_shift":
        qq[:, 2] = .5
    a = rule_activations(qq)
    beliefs = expit(head["parameters"])
    return a @ beliefs if head["kind"] == "sugeno" else rimer_correct(a, beliefs)


def _fit_head(kind, q, target, weights):
    prior = float((np.sum(target) + .5) / (len(target) + 1))
    if min(int(target.sum()), int((1 - target).sum())) < MIN_EVENTS:
        return {"kind": "constant", "requested_kind": kind, "value": prior,
                "fallback": "fewer than five correct or incorrect examples"}
    if kind == "logistic":
        x = np.column_stack([np.ones(len(q)), q - .5])
        center = np.array([logit(prior), 0, 0, 0])
        def objective(theta):
            raw = expit(x @ theta)
            loss = _binary_loss(target, raw, weights) + HEAD_L2 * np.mean((theta - center) ** 2)
            grad = x.T @ (weights * (raw - target)) + 2 * HEAD_L2 * (theta - center) / len(theta)
            return loss, grad
    else:
        qq = q.copy()
        if kind == "brb_no_shift":
            qq[:, 2] = .5
        a = rule_activations(qq)
        center = np.full(8, logit(prior))
        def objective(theta):
            beliefs = expit(theta)
            if kind == "sugeno":
                raw, jac = a @ beliefs, a
            else:
                raw, jac = rimer_correct(a, beliefs, return_jacobian=True)
            pp = np.clip(raw, EPS, 1 - EPS)
            derivative = weights * (pp - target) / (pp * (1 - pp))
            grad = (derivative @ jac) * beliefs * (1 - beliefs)
            loss = _binary_loss(target, pp, weights) + HEAD_L2 * np.mean((theta - center) ** 2)
            grad += 2 * HEAD_L2 * (theta - center) / len(theta)
            return loss, grad
    opt = minimize(objective, center, jac=True, method="L-BFGS-B", bounds=[(-10, 10)] * len(center),
                   options={"maxiter": MAXITER, "ftol": 1e-9})
    if not np.isfinite(opt.fun) or not np.isfinite(opt.x).all():
        raise RuntimeError(f"nonfinite {kind} fit")
    return {"kind": kind, "parameters": opt.x.tolist(), "optimization": _optimization(opt), "prior": prior}


def _fit_calibration(raw, target):
    """Monotone logit-affine reliability calibration on repetition 6 only."""
    if len(target) < 20:
        return {"slope": 1., "intercept": 0., "fallback": "fewer than 20 calibration rows"}
    x = logit(np.clip(raw, 1e-5, 1 - 1e-5))
    sparse = min(int(target.sum()), int((1 - target).sum())) < MIN_EVENTS
    def objective(theta):
        slope, intercept = theta
        p = expit(slope * x + intercept)
        weights = np.full(len(target), 1 / len(target))
        loss = _binary_loss(target, p, weights) + CAL_L2 * ((slope - 1) ** 2 + intercept ** 2)
        residual = p - target
        grad = np.array([np.mean(residual * x) + 2 * CAL_L2 * (slope - 1),
                         np.mean(residual) + 2 * CAL_L2 * intercept])
        return loss, grad
    opt = minimize(objective, [1., 0.], jac=True, method="L-BFGS-B",
                   bounds=[(1., 1.) if sparse else (0., 5.), (-8., 8.)], options={"maxiter": MAXITER})
    return {"slope": float(opt.x[0]), "intercept": float(opt.x[1]), "optimization": _optimization(opt),
            "fallback": "intercept-only: fewer than five events in one outcome" if sparse else None}


def _calibrated(raw, calibration):
    return expit(calibration["slope"] * logit(np.clip(raw, 1e-5, 1 - 1e-5)) + calibration["intercept"])


def _weights(alpha, reliability):
    unnormalized = np.asarray(alpha)[None, :] * np.clip(reliability, 0, 1)
    denominator = unnormalized.sum(1, keepdims=True)
    return np.divide(unnormalized, denominator, out=np.broadcast_to(alpha, unnormalized.shape).copy(), where=denominator > EPS)


def predict_diagnostics(model, logits, shifts):
    z, s = _inputs(logits, shifts)
    if model.get("version") != VERSION:
        raise ValueError("unsupported meta model version")
    p = _probabilities(z, model["temperatures"])
    q = indicators(p, s)
    reliabilities = {"confidence_weight": p.max(2)}
    raw_reliabilities = {}
    for name, heads in model["heads"].items():
        raw = np.column_stack([_raw_head(heads[j], q[:, j]) for j in range(3)])
        raw_reliabilities[name] = raw
        reliabilities[name] = np.column_stack([_calibrated(raw[:, j], model["reliability_calibrations"][name][j]) for j in range(3)])
    weights = {name: _weights(model["alpha"], r) for name, r in reliabilities.items()}
    return {"expert_probabilities": p, "indicators": q, "raw_reliabilities": raw_reliabilities,
            "reliabilities": reliabilities, "weights": weights, "rule_activations": rule_activations(q)}


def predict_meta(model, logits, shifts):
    d = predict_diagnostics(model, logits, shifts)
    p = d["expert_probabilities"]
    result = {f"expert_{name.lower()}": p[:, j] for j, name in enumerate(EXPERTS)}
    result["mean"] = p.mean(1)
    result["global_weight"] = np.sum(p * np.asarray(model["alpha"])[None, :, None], axis=1)
    result.update({name: np.sum(p * w[:, :, None], axis=1) for name, w in d["weights"].items()})
    return result


def _metrics(y, p):
    confidence, predicted = p.max(1), p.argmax(1)
    correct = predicted == y
    onehot = np.eye(p.shape[1])[y]
    ece = 0.
    for low in np.arange(0, 1, .1):
        mask = (confidence >= low) & (confidence < low + .1 if low < .9 else confidence <= 1)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return {"rows": int(len(y)), "accuracy": float(correct.mean()),
            "nll": float(-np.log(np.clip(p[np.arange(len(y)), y], EPS, 1)).mean()),
            "brier": float(np.sum((p - onehot) ** 2, axis=1).mean()), "ece10": float(ece)}


def _write_csv(path, rows):
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def fit_meta(logits, y, repetitions, shifts, output: Path, metadata=None):
    """Fit and save a JSON-serializable meta model; never supply test rows.

    1/3/4: final temperatures, alpha, reliability heads. Head training inputs use
    leave-one-repetition crossfitted temperatures. 6: monotone scalar reliability
    calibration only. Final temperature is frozen before looking at rep 6.
    """
    z, s, yy, rr = _inputs(logits, shifts, y, repetitions)
    fit = np.isin(rr, FIT_REPS)
    cal = rr == CAL_REP
    temperatures, tdiag = fit_temperatures(z[fit], yy[fit], rr[fit])
    p_final = _probabilities(z, temperatures)
    p_crossfit = np.zeros_like(z[fit])
    crossfits = []
    for held in FIT_REPS:
        inner_train = fit & (rr != held)
        temp, opt = fit_temperatures(z[inner_train], yy[inner_train], rr[inner_train])
        p_crossfit[rr[fit] == held] = _probabilities(z[fit & (rr == held)], temp)
        crossfits.append({"held_repetition": held, "fit_repetitions": sorted(set(rr[inner_train].tolist())),
                          "temperatures": temp, "optimization": opt})
    q_fit = indicators(p_crossfit, s[fit])
    q_cal = indicators(p_final[cal], s[cal])
    # Positive temperature cannot change an expert's class argmax.
    correctness_fit = (z[fit].argmax(2) == yy[fit, None]).astype(float)
    correctness_cal = (z[cal].argmax(2) == yy[cal, None]).astype(float)
    weights_fit = _group_weights(rr[fit])
    alpha, adiag = _fit_alpha(p_crossfit, yy[fit], weights_fit)
    model = {"version": VERSION, "expert_order": list(EXPERTS), "num_classes": 17,
             "temperatures": temperatures, "alpha": alpha, "heads": {}, "reliability_calibrations": {},
             "metadata": metadata or {}, "provenance": {
                 "meta_fit_repetitions": list(FIT_REPS), "reliability_calibration_repetition": CAL_REP,
                 "outer_test_repetitions_forbidden_at_fit": [2, 5], "fit_rows": int(fit.sum()), "calibration_rows": int(cal.sum()),
                 "counts_per_repetition": {str(r): int((rr == r).sum()) for r in sorted(set(rr.tolist()))},
                 "temperatures_crossfit": crossfits, "final_temperature_optimization": tdiag, "alpha_optimization": adiag,
                 "alpha_fit_input": "crossfitted-temperature probabilities from repetitions 1/3/4",
                 "fit_digest": hashlib.sha256(z[fit].tobytes() + yy[fit].tobytes() + s[fit].tobytes() + rr[fit].tobytes()).hexdigest(),
                 "calibration_digest": hashlib.sha256(z[cal].tobytes() + yy[cal].tobytes() + s[cal].tobytes()).hexdigest(),
                 "fixed_hyperparameters": {"head_l2": HEAD_L2, "calibration_l2": CAL_L2, "alpha_l2": ALPHA_L2, "min_events": MIN_EVENTS},
                 "rule_inputs": ["normalized_entropy", "mean_pairwise_total_variation", "training_reference_deviation"],
                 "caveat": "internal development; shared base-model training histories mean this is not independent meta CV"}}
    support_rows, rule_rows, reliability_rows = [], [], []
    for name in ("logistic", "sugeno", "brb", "brb_no_shift"):
        model["heads"][name], model["reliability_calibrations"][name] = [], []
        for j, expert in enumerate(EXPERTS):
            head = _fit_head(name, q_fit[:, j], correctness_fit[:, j], weights_fit)
            raw_cal = _raw_head(head, q_cal[:, j])
            calibration = _fit_calibration(raw_cal, correctness_cal[:, j])
            model["heads"][name].append(head)
            model["reliability_calibrations"][name].append(calibration)
            for stage, values in [("before", raw_cal), ("after", _calibrated(raw_cal, calibration))]:
                reliability_rows.append({"method": name, "expert": expert, "stage": stage, "repetition": 6,
                    "scope": "calibration fitting rows; not independent evaluation", "rows": len(raw_cal),
                    "observed_correctness": float(correctness_cal[:, j].mean()), "mean_reliability": float(values.mean()),
                    "brier_binary": float(np.mean((values - correctness_cal[:, j]) ** 2)),
                    "nll_binary": _binary_loss(correctness_cal[:, j], values, np.full(len(values), 1 / len(values)))})
            if name != "logistic":
                qq = q_fit[:, j].copy()
                if name == "brb_no_shift":
                    qq[:, 2] = .5
                a = rule_activations(qq)
                beliefs = np.full(8, head["value"]) if head["kind"] == "constant" else expit(head["parameters"])
                for rule, bits in enumerate(RULE_BITS):
                    rule_rows.append({"method": name, "expert": expert, "rule": rule,
                        "uncertainty": int(bits[0]), "disagreement": int(bits[1]), "deviation": int(bits[2]),
                        "belief_incorrect": float(1 - beliefs[rule]), "belief_correct": float(beliefs[rule])})
                    for rep in FIT_REPS:
                        mask = rr[fit] == rep
                        mass = a[mask, rule]
                        support_rows.append({"method": name, "expert": expert, "rule": rule, "repetition": rep,
                            "window_count": int(mask.sum()), "activation_mass": float(mass.sum()),
                            "mean_activation": float(mass.mean()), "windows_activation_above_0_1": int((mass > .1).sum()),
                            "effective_windows_kish_correlated_not_trials": float(mass.sum() ** 2 / max(np.sum(mass ** 2), EPS)),
                            "weighted_correctness": float(mass @ correctness_fit[mask, j] / max(mass.sum(), EPS))})
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    (output / "meta_model.json").write_text(json.dumps(model, indent=2, allow_nan=False), encoding="utf-8")
    (output / "meta_provenance.json").write_text(json.dumps(model["provenance"], indent=2, allow_nan=False), encoding="utf-8")
    _write_csv(output / "meta_rule_conclusions.csv", rule_rows)
    _write_csv(output / "meta_rule_support_per_repetition.csv", support_rows)
    _write_csv(output / "meta_reliability_calibration.csv", reliability_rows)
    pred = predict_meta(model, z, s)
    metrics = [{"method": name, "repetition": int(rep),
                "scope": "meta-head development fit" if rep in FIT_REPS else "reliability calibration fit",
                **_metrics(yy[rr == rep], pp[rr == rep])}
               for name, pp in pred.items() for rep in sorted(set(rr.tolist()))]
    _write_csv(output / "meta_development_metrics.csv", metrics)
    return model


### brb_worker.py

One subject worker: four OOF folds, locked meta fitting, final models and test diagnostics.


In [ ]:
%%writefile /kaggle/working/db7_brb_src/brb_worker.py
"""Isolated-GPU neural/OOF worker for the DB7 BRB pilot.

Each subject/seed has 12 leave-one-training-repetition-out expert fits and five
final fits. Calibration and reliability learning finish before test inference.
The unchanged TC loader supplies physical segments; its convenience all-four
input scaler is explicitly ignored and every fold computes its own statistics.
"""
from __future__ import annotations

import argparse
import gc
import hashlib
import importlib
import json
import os
from pathlib import Path
import time
import traceback

import numpy as np
import pandas as pd

TRAIN_REPS = (1, 3, 4, 6)
TEST_REPS = (2, 5)
EXPERTS = ("W", "S", "I")
FINAL_ARMS = (*EXPERTS, "SI", "WSI")
KEYS = ["subject", "gesture", "native_repetition", "window_start", "window_end"]


def json_write(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"),
                                     allow_nan=False).encode()).hexdigest()


def file_hash(path):
    result = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            result.update(block)
    return result.hexdigest()


def window_hash(meta):
    return hashlib.sha256(meta[KEYS].to_csv(index=False).encode()).hexdigest()


def training_signal_hash(segments, metadata):
    """Bind checkpoints to physical training samples, not just their moments."""
    if set(metadata["split"]) != {"train"} or not set(metadata.native_repetition).issubset(TRAIN_REPS):
        raise ValueError("Training signal hash requires development-only train metadata")
    result = hashlib.sha256()
    for segment in sorted(map(int, metadata.segment.unique())):
        values = np.ascontiguousarray(segments[segment])
        result.update(json.dumps([segment, list(values.shape), str(values.dtype)]).encode())
        result.update(memoryview(values).cast("B"))
    return result.hexdigest()


def atomic_npz(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("wb") as handle:
        np.savez_compressed(handle, **arrays)
    temporary.replace(path)


def validate_config(config):
    """Fail rather than silently modifying the accepted pilot protocol."""
    fixed = dict(window_samples=400, stride_samples=20, fs=2000, batch_size=512,
                 dropout=0.65, epochs=2 if config.get("smoke", False) else 13)
    for key, value in fixed.items():
        if config.get(key, value) != value:
            raise ValueError(f"Unexpected {key}: this pilot requires {value}")
    if tuple(config.get("train_repetitions", TRAIN_REPS)) != TRAIN_REPS:
        raise ValueError("Development repetitions must be 1/3/4/6")
    if tuple(config.get("test_repetitions", TEST_REPS)) != TEST_REPS:
        raise ValueError("Outer test repetitions must be 2/5")
    if config.get("reference_cap_per_trial", 128) < 1:
        raise ValueError("Reference cap must be positive")
    if not config.get("subjects") or not config.get("seeds"):
        raise ValueError("Explicit subjects and seeds are required")
    if config.get("smoke", False):
        if config.get("smoke_windows_per_trial", 32) < 1:
            raise ValueError("Smoke windows per trial must be positive")
        if config.get("smoke_max_train_batches", 2) < 1:
            raise ValueError("Smoke training-batch cap must be positive")


def fold_inventory(inventory, training_repetitions, evaluation_repetitions):
    """Copy the inventory and assign splits solely from native repetition IDs."""
    train = set(map(int, training_repetitions))
    evaluation = set(map(int, evaluation_repetitions))
    if not train or not train.issubset(TRAIN_REPS):
        raise ValueError("Only development repetitions can train a model")
    if not evaluation or train.intersection(evaluation):
        raise ValueError("Training and evaluation repetitions must be disjoint")
    if not evaluation.issubset(set(TRAIN_REPS) | set(TEST_REPS)):
        raise ValueError("Unknown evaluation repetition")
    result = inventory.copy()
    result["split"] = "excluded"
    result.loc[result.native_repetition.isin(train), "split"] = "train"
    result.loc[result.native_repetition.isin(evaluation), "split"] = "evaluation"
    for split in ("train", "evaluation"):
        selected = result.loc[result.split == split]
        if set(selected.gesture) != set(range(1, 18)):
            raise ValueError(f"{split}: all 17 gestures are required")
        if selected.duplicated(["subject", "gesture", "native_repetition"]).any():
            raise ValueError("Expected one segment per subject/gesture/repetition")
        expected = train if split == "train" else evaluation
        for _, group in selected.groupby("gesture"):
            if set(group.native_repetition) != expected:
                raise ValueError("Gesture repetition coverage is incomplete")
    return result


def fit_input_scaler(segments, inventory, training_repetitions):
    """Every physical training sample counts once, independent of window overlap."""
    expected = set(map(int, training_repetitions))
    rows = inventory.loc[inventory.split == "train"]
    if not expected or not expected.issubset(TRAIN_REPS):
        raise ValueError("Scaler training repetitions must be development-only")
    if set(rows.native_repetition) != expected:
        raise ValueError("Scaler inventory provenance does not match this fold")
    total = np.zeros(120, dtype=np.float64)
    m2 = np.zeros(120, dtype=np.float64)
    count = 0
    for row in rows.to_dict("records"):
        values = np.asarray(segments[int(row["segment"])] , dtype=np.float64)
        if values.ndim != 2 or values.shape[1] != 120 or not np.isfinite(values).all():
            raise ValueError("Invalid physical training segment")
        batch_count = len(values)
        if batch_count == 0:
            raise ValueError("Empty training segment")
        batch_mean = values.mean(axis=0)
        combined = count + batch_count
        delta = batch_mean - total
        m2 += ((values - batch_mean) ** 2).sum(axis=0) + delta ** 2 * count * batch_count / combined
        total += delta * batch_count / combined
        count = combined
    if not count:
        raise ValueError("Empty scaler training population")
    std = np.sqrt(np.maximum(m2 / count, 1e-12))
    provenance = dict(training_repetitions=sorted(expected), training_sample_count=count,
                      training_segments=len(rows), statistic="population_sample_moments",
                      variance_floor=1e-12,
                      inventory_sha256=hashlib.sha256(rows.to_csv(index=False).encode()).hexdigest())
    return total.astype(np.float32), std.astype(np.float32), provenance


def balanced_indices(metadata, cap):
    """Use equal deterministic, time-spread window counts for every trial."""
    groups = list(metadata.groupby(["subject", "gesture", "native_repetition"], sort=True).indices.values())
    if not groups or cap < 1:
        raise ValueError("Nonempty metadata and positive cap are required")
    count = min(int(cap), min(map(len, groups)))
    return np.sort(np.concatenate([indices[np.linspace(0, len(indices) - 1, count).round().astype(int)]
                                   for indices in groups]))


def make_windows(segments, inventory, mean, std, split, smoke=False, smoke_cap=32):
    support = importlib.import_module("tc_support")
    dataset = support.TCWindows(segments, inventory, mean, std, split)
    if not len(dataset):
        raise ValueError("No windows were constructed")
    if smoke:
        keep = balanced_indices(dataset.meta, smoke_cap)
        dataset.meta = dataset.meta.iloc[keep].reset_index(drop=True)
        dataset.y = dataset.y[keep]
        dataset.locations = dataset.locations[keep]
    if set(dataset.y) != set(range(17)):
        raise ValueError("All 17 classes must occur even in smoke mode")
    return dataset


def raw_descriptors(windows):
    """Physical features; no learned transform and no labels used here.

    Input [N,120,400], output RMS12, Hann-FFT mean-frequency12, inertial mean108.
    The feature FFT deliberately covers the entire 200 ms outer window and is
    separate from the model's 100 ms frame FFT. Frequencies include 20..450 Hz.
    """
    values = np.asarray(windows, dtype=np.float64)
    if values.ndim != 3 or values.shape[1:] != (120, 400) or not np.isfinite(values).all():
        raise ValueError("Expected finite physical windows [N,120,400]")
    emg = values[:, :12]
    rms = np.sqrt(np.square(emg).mean(axis=-1))
    frequencies = np.fft.rfftfreq(400, d=1 / 2000)
    selected = (frequencies >= 20) & (frequencies <= 450)
    power = np.abs(np.fft.rfft(emg * np.hanning(400), axis=-1)) ** 2
    power = power[:, :, selected]
    total = power.sum(axis=-1)
    mean_frequency = np.divide((power * frequencies[selected]).sum(axis=-1), total,
                               out=np.zeros_like(total), where=total > 0)
    means = values[:, 12:].mean(axis=-1)
    return np.concatenate([rms, mean_frequency, means], axis=1)


def dataset_descriptors(dataset):
    arrays = []
    for begin in range(0, len(dataset), 128):
        values = np.stack([dataset.segments[int(segment)][int(offset):int(offset) + 400].T
                           for segment, offset in dataset.locations[begin:begin + 128]])
        arrays.append(raw_descriptors(values))
    result = np.concatenate(arrays)
    if result.shape != (len(dataset), 132):
        raise ValueError("Physical feature dimensionality mismatch")
    return result


def feature_columns():
    return ([f"emg_{channel:02}_rms" for channel in range(1, 13)] +
            [f"emg_{channel:02}_mean_frequency" for channel in range(1, 13)] +
            [f"{sensor}_{channel:02}_mean" for sensor in ("acc", "gyro", "mag")
             for channel in range(1, 37)])


def fit_shift_reference(raw_features, metadata, training_repetitions, cap=128):
    """Fit log floor, robust centers and scales using only this fold's train trials."""
    raw_features = np.asarray(raw_features, dtype=np.float64)
    if raw_features.shape != (len(metadata), 132) or not np.isfinite(raw_features).all():
        raise ValueError("Invalid training physical features")
    expected = set(map(int, training_repetitions))
    if not expected.issubset(TRAIN_REPS) or set(metadata.native_repetition) != expected:
        raise ValueError("Reference features contain unexpected or test repetitions")
    if set(metadata["split"]) != {"train"}:
        raise ValueError("Reference feature metadata must be explicitly training")
    keep = balanced_indices(metadata, cap)
    selected = raw_features[keep]
    # Every floor is fitted from the same capped, equally weighted training trials.
    rms_floor = np.maximum(selected[:, :12].max(axis=0) * 1e-6, 1e-12)
    transformed = selected.copy()
    transformed[:, :12] = np.log(np.maximum(transformed[:, :12], rms_floor))
    center = np.median(transformed, axis=0)
    q25, q75 = np.percentile(transformed, [25, 75], axis=0)
    iqr = q75 - q25
    scale_floor = np.maximum(np.abs(transformed).max(axis=0) * 1e-6, 1e-12)
    scale = np.maximum(iqr, scale_floor)
    reference = dict(version=1, training_repetitions=sorted(expected),
                     available_training_windows=len(metadata), selected_training_windows=len(keep),
                     trials=int(metadata.groupby(["subject", "gesture", "native_repetition"]).ngroups),
                     requested_cap_per_trial=int(cap), effective_windows_per_trial=len(keep) //
                     int(metadata.groupby(["subject", "gesture", "native_repetition"]).ngroups),
                     selected_window_sha256=window_hash(metadata.iloc[keep]),
                     source_feature_sha256=hashlib.sha256(np.ascontiguousarray(selected).tobytes()).hexdigest(),
                     feature_columns=feature_columns(), rms_floor=rms_floor.tolist(),
                     center=center.tolist(), scale=scale.tolist(), iqr=iqr.tolist(),
                     scale_floor=scale_floor.tolist(),
                     formula="mean(abs((feature-center)/scale))/(1+mean(abs((feature-center)/scale)))",
                     waveform_transform="natural_log(max(physical_rms, training_rms_floor))",
                     spectral_descriptor="mean_frequency: symmetric_Hann_FFT400_20_to_450Hz",
                     arm_feature_slices={"W": [0, 12], "S": [12, 24], "I": [24, 132]})
    reference["sha256"] = digest(reference)
    return reference, keep


def apply_shift_reference(raw_features, reference):
    values = np.asarray(raw_features, dtype=np.float64).copy()
    if values.ndim != 2 or values.shape[1] != 132 or not np.isfinite(values).all():
        raise ValueError("Invalid physical feature array")
    values[:, :12] = np.log(np.maximum(values[:, :12], reference["rms_floor"]))
    deviations = np.abs((values - reference["center"]) / reference["scale"])
    shifts = np.column_stack([deviations[:, begin:end].mean(axis=1)
                              for begin, end in ((0, 12), (12, 24), (24, 132))])
    shifts = shifts / (1 + shifts)
    if not np.isfinite(shifts).all() or np.any(shifts < 0) or np.any(shifts > 1):
        raise ValueError("Invalid normalized shift")
    return shifts.astype(np.float32)


def probability_metrics(probabilities, y):
    p = np.asarray(probabilities, dtype=np.float64)
    y = np.asarray(y, dtype=int)
    if p.shape != (len(y), 17) or not np.isfinite(p).all() or np.any(p < 0):
        raise ValueError("Invalid 17-class probability output")
    if not np.allclose(p.sum(axis=1), 1, atol=1e-5) or not len(y):
        raise ValueError("Invalid probability normalization or empty targets")
    if y.min() < 0 or y.max() > 16:
        raise ValueError("Expected zero-based gesture targets")
    predicted = p.argmax(axis=1)
    correct = predicted == y
    confusion = np.zeros((17, 17), dtype=np.int64)
    np.add.at(confusion, (y, predicted), 1)
    f1 = 2 * confusion.diagonal() / np.maximum(confusion.sum(0) + confusion.sum(1), 1)
    recall = confusion.diagonal() / np.maximum(confusion.sum(1), 1)
    onehot = np.eye(17)[y]
    confidence = p.max(axis=1)
    ece = 0.0
    for lower in np.arange(15) / 15:
        selection = (confidence >= lower) & ((confidence < lower + 1 / 15) if lower < 14 / 15 else (confidence <= 1))
        if selection.any():
            ece += selection.mean() * abs(confidence[selection].mean() - correct[selection].mean())
    return dict(n_windows=len(y), correct=int(correct.sum()), wrong=int((~correct).sum()),
                accuracy=float(correct.mean()), macro_f1=float(f1.mean()),
                balanced_accuracy=float(recall.mean()),
                nll=float(-np.log(p[np.arange(len(y)), y].clip(1e-12)).mean()),
                brier=float(np.square(p - onehot).sum(axis=1).mean()), ece15=float(ece))


def save_predictions(folder, dataset, logits):
    logits = np.asarray(logits, dtype=np.float32)
    if logits.shape != (len(dataset), 17) or not np.isfinite(logits).all():
        raise ValueError("Invalid logits")
    shifted = logits.astype(np.float64) - logits.max(axis=1, keepdims=True)
    probs = np.exp(shifted)
    probs /= probs.sum(axis=1, keepdims=True)
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    atomic_npz(folder / "predictions.npz", logits=logits, probabilities=probs.astype(np.float32), y=dataset.y)
    frame = dataset.meta.copy()
    frame["prediction"] = probs.argmax(axis=1) + 1
    frame["correct"] = frame.prediction.to_numpy() == frame.gesture.to_numpy()
    frame["confidence"] = probs.max(axis=1)
    frame.to_csv(folder / "predictions.csv", index=False)
    json_write(folder / "metrics.json", probability_metrics(probs, dataset.y))
    return probs.astype(np.float32)


def model_predict(model, dataset, batch_size):
    import torch
    from torch.utils.data import DataLoader
    model.eval()
    pieces = []
    with torch.no_grad():
        for x, _ in DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0):
            pieces.append(model(x.cuda()).cpu().numpy())
    return np.concatenate(pieces)


def train_fit(arm, seed, subject, train, folder, fold_tag, run_signature, config):
    """Return the final fixed-epoch model; exact completed fits may be reused."""
    import torch
    from torch.nn import functional as F
    from torch.utils.data import DataLoader
    from ablation_model import AblationC1
    import tc_support as support
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    smoke = bool(config.get("smoke", False))
    epochs = 2 if smoke else 13
    actual_seed = int(seed) + 1000 * int(subject)
    fit_spec = dict(run_signature=run_signature, subject=int(subject), seed=int(seed),
                    actual_seed=actual_seed, arm=arm, fold=fold_tag, epochs=epochs,
                    stage="oof" if fold_tag.startswith("oof_") else "final",
                    held_out_repetition=int(fold_tag.rsplit("_", 1)[1]) if fold_tag.startswith("oof_") else None,
                    training_window_sha256=window_hash(train.meta),
                    training_signal_sha256=training_signal_hash(train.segments, train.meta),
                    input_scaler_sha256=hashlib.sha256(train.mean.tobytes() + train.std.tobytes()).hexdigest(),
                    training_repetitions=sorted(map(int, train.meta.native_repetition.unique())),
                    training_windows=len(train), smoke=smoke)
    signature = digest(fit_spec)
    manifest_path = folder / "fit_manifest.json"
    checkpoint = folder / "final_model.pt"
    if manifest_path.exists():
        prior = json.loads(manifest_path.read_text(encoding="utf-8"))
        if prior.get("signature") != signature:
            raise RuntimeError(f"Refusing incompatible fit reuse: {folder}")
        if prior.get("success"):
            if not checkpoint.exists() or file_hash(checkpoint) != prior.get("checkpoint_sha256"):
                raise RuntimeError("Completed checkpoint missing or corrupted")
            model = AblationC1(arm, config.get("dropout", 0.65)).cuda()
            model.load_state_dict(torch.load(checkpoint, map_location="cuda", weights_only=True))
            print(f"RESUME verified {folder}", flush=True)
            return model
    json_write(manifest_path, dict(**fit_spec, signature=signature, success=False,
                                  state="training", status="training", epochs_run=0,
                                  metric_usage="no evaluation checkpoint selection"))
    support.set_seed(actual_seed)
    model = AblationC1(arm, config.get("dropout", 0.65)).cuda()
    batch_size = int(config.get("batch_size", 512))
    if "S" in arm:
        scaler_info = model.fit_spectral_scaler(DataLoader(train, batch_size=batch_size, shuffle=False,
                                                           num_workers=0), split="train")
        scaler_info.update(training_window_sha256=fit_spec["training_window_sha256"],
                           training_repetitions=fit_spec["training_repetitions"], fold=fold_tag)
        json_write(folder / "spectral_scaler.json", scaler_info)
    support.set_seed(actual_seed)
    loader = DataLoader(train, batch_size=batch_size, shuffle=True, num_workers=0,
                        generator=torch.Generator().manual_seed(actual_seed + 12345))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=0)
    history = []
    torch.cuda.reset_peak_memory_stats()
    for epoch in range(1, epochs + 1):
        lr = 1e-3 if epoch <= 3 else 1e-4 if epoch <= 9 else 1e-5
        for group in optimizer.param_groups:
            group["lr"] = lr
        model.train()
        total = correct = steps = 0
        loss_sum = 0.0
        started = time.perf_counter()
        for batch_index, (x, y) in enumerate(loader):
            if smoke and batch_index >= int(config.get("smoke_max_train_batches", 2)):
                break
            x, y = x.cuda(), y.cuda()
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            if not torch.isfinite(loss):
                raise ValueError("Nonfinite neural training loss")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
            optimizer.step()
            total += len(y)
            correct += int((logits.argmax(1) == y).sum())
            loss_sum += float(loss.detach()) * len(y)
            steps += 1
        history.append(dict(epoch=epoch, learning_rate=lr, train_loss=loss_sum / total,
                            train_accuracy=correct / total, optimizer_steps=steps,
                            examples_seen=total, seconds=time.perf_counter() - started))
        pd.DataFrame(history).to_csv(folder / "history.csv", index=False)
        print(f"GPU{os.environ.get('CUDA_VISIBLE_DEVICES')} S{subject:02} seed{seed} {fold_tag} "
              f"{arm} epoch{epoch}/{epochs} train={correct / total:.4f}", flush=True)
    temporary = checkpoint.with_name(checkpoint.name + ".tmp")
    torch.save(model.state_dict(), temporary)
    temporary.replace(checkpoint)
    json_write(manifest_path, dict(**fit_spec, signature=signature, success=True, state="fit_complete",
                                  status="complete", epochs_run=epochs,
                                  checkpoint_sha256=file_hash(checkpoint),
                                  parameter_count=int(model.count_params()),
                                  gpu=os.environ.get("CUDA_VISIBLE_DEVICES"), gpu_name=torch.cuda.get_device_name(0),
                                  peak_cuda_memory_bytes=int(torch.cuda.max_memory_allocated()),
                                  optimizer="Adam", weight_decay=0, gradient_clip=5,
                                  augmentation=False, checkpoint_selection="fixed_final_epoch",
                                  training_seconds=sum(row["seconds"] for row in history),
                                  smoke_label="NONRESEARCH SMOKE" if smoke else None))
    del optimizer, loader
    return model


def gpu_preflight(gpu, output):
    import torch
    from ablation_model import AblationC1
    if torch.cuda.device_count() != 1:
        raise RuntimeError("Worker requires exactly one visible CUDA GPU, pinned by launcher before import")
    if os.environ.get("CUDA_VISIBLE_DEVICES") != str(gpu):
        raise RuntimeError("CUDA_VISIBLE_DEVICES does not match assigned physical GPU")
    rows = []
    for arm in EXPERTS:
        model = AblationC1(arm, 0.65).cuda()
        x = torch.randn(2, 120, 400, device="cuda")
        if "S" in arm:
            model.fit_spectral_scaler([x])
        logits = model(x)
        loss = torch.nn.functional.cross_entropy(logits, torch.tensor([0, 16], device="cuda"))
        loss.backward()
        if logits.shape != (2, 17) or not torch.isfinite(logits).all():
            raise RuntimeError("Neural shape preflight failed")
        if not all(parameter.grad is not None and torch.isfinite(parameter.grad).all()
                   for parameter in model.parameters()):
            raise RuntimeError("Neural gradient preflight failed")
        rows.append(dict(arm=arm, shape=list(logits.shape), gradients_finite=True))
        del model, x, logits, loss
    torch.cuda.empty_cache()
    json_write(Path(output) / f"gpu{gpu}_preflight.json",
               dict(success=True, physical_gpu=gpu, visible_devices=os.environ["CUDA_VISIBLE_DEVICES"],
                    gpu_name=torch.cuda.get_device_name(0), pid=os.getpid(), checks=rows))


def metadata_indices(full_meta, subset_meta):
    full_index = pd.MultiIndex.from_frame(full_meta[KEYS])
    query = pd.MultiIndex.from_frame(subset_meta[KEYS])
    if not full_index.is_unique or not query.is_unique:
        raise ValueError("Duplicate window identity")
    selected = full_index.get_indexer(query)
    if (selected < 0).any():
        raise ValueError("Subset window identity absent from feature cache")
    return selected


def export_meta_results(folder, metadata, y, predictions, diagnostics, si_probs, wsi_probs):
    """Explicit window/trial/gesture/phase tables for all fusion methods and controls."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    all_predictions = dict(predictions, control_si=si_probs, control_wsi=wsi_probs)
    metrics, cases = [], []
    baseline = si_probs.argmax(axis=1)
    for method, probabilities in all_predictions.items():
        probabilities = np.asarray(probabilities)
        row = dict(method=method, **probability_metrics(probabilities, y))
        row["probability_source"] = "uncalibrated_control_softmax" if method.startswith("control_") else "calibrated_expert_fusion"
        for column in ("subject", "seed"):
            if column in metadata and metadata[column].nunique() == 1:
                row[column] = int(metadata[column].iloc[0])
        predicted = probabilities.argmax(axis=1)
        row["recovered_vs_si"] = int(((baseline != y) & (predicted == y)).sum())
        row["harmed_vs_si"] = int(((baseline == y) & (predicted != y)).sum())
        metrics.append(row)
        cases.append(metadata.assign(method=method, prediction=predicted + 1,
                                     correct=predicted == y, confidence=probabilities.max(axis=1),
                                     recovered_vs_si=(baseline != y) & (predicted == y),
                                     harmed_vs_si=(baseline == y) & (predicted != y)))
        confusion = pd.crosstab(pd.Series(y + 1, name="gesture"),
                                pd.Series(predicted + 1, name="prediction")).reindex(
                                    index=range(1, 18), columns=range(1, 18), fill_value=0)
        confusion.to_csv(folder / f"confusion_{method}.csv")
    frame = pd.concat(cases, ignore_index=True)
    frame.to_csv(folder / "per_window_predictions.csv", index=False)
    pd.DataFrame(metrics).to_csv(folder / "method_metrics.csv", index=False)
    for name, keys in [("subject", ["subject"]), ("gesture", ["subject", "gesture"]),
                       ("repetition", ["subject", "gesture", "native_repetition"]),
                       ("phase", ["subject", "gesture", "native_repetition", "phase"])]:
        summary = frame.groupby(["method"] + keys, dropna=False).agg(
            windows=("correct", "size"), correct=("correct", "sum"),
            recovered_vs_si=("recovered_vs_si", "sum"), harmed_vs_si=("harmed_vs_si", "sum")).reset_index()
        summary["wrong"] = summary.windows - summary.correct
        summary["accuracy"] = summary.correct / summary.windows
        summary.to_csv(folder / f"per_{name}_errors.csv", index=False)
    atomic_npz(folder / "fusion_probabilities.npz", **{key: np.asarray(value, np.float32)
                                                      for key, value in all_predictions.items()}, y=y)
    flat_diagnostics = {}
    def flatten(prefix, value):
        if isinstance(value, dict):
            for key, child in value.items():
                flatten(f"{prefix}_{key}" if prefix else key, child)
        elif isinstance(value, (np.ndarray, list, tuple)):
            array = np.asarray(value)
            if array.dtype.kind in "biuf" and array.ndim and array.shape[0] == len(y):
                flat_diagnostics[prefix] = array
    flatten("", diagnostics)
    atomic_npz(folder / "reliability_diagnostics.npz", **flat_diagnostics)
    return metrics


def run_subject(subject, seed, config, output, run_signature):
    import torch
    from ablation_model import AblationC1
    import tc_support as support
    import brb_meta
    root = Path(output) / f"S{subject:02}" / f"seed_{seed}"
    root.mkdir(parents=True, exist_ok=True)
    raw_root = root / "source_inventory"
    raw_root.mkdir(exist_ok=True)
    source = Path(config["input_root"]) if config.get("input_root") else support.find_input()
    segments, inventory, _ignored_mean, _ignored_std = support.load_subject(source, subject, raw_root)
    json_write(raw_root / "SCALER_NOT_USED.json", dict(
        explanation="tc_support.load_subject convenience scaler is never used by BRB worker; fold scalers are recomputed",
        unused_scaler="input_scaler.npz"))
    smoke = bool(config.get("smoke", False))
    window_options = dict(smoke=smoke, smoke_cap=int(config.get("smoke_windows_per_trial", 32)))
    complete_inventory = fold_inventory(inventory, TRAIN_REPS, TEST_REPS)
    final_mean, final_std, final_provenance = fit_input_scaler(segments, complete_inventory, TRAIN_REPS)
    development = make_windows(segments, complete_inventory, final_mean, final_std, "train", **window_options)
    development.meta.to_csv(root / "development_windows.csv", index=False)
    development_raw = dataset_descriptors(development)
    atomic_npz(root / "development_physical_features.npz", features=development_raw)
    oof_logits = np.full((len(development), 3, 17), np.nan, dtype=np.float32)
    oof_shifts = np.full((len(development), 3), np.nan, dtype=np.float32)
    covered = np.zeros(len(development), dtype=int)
    fold_records = []
    cap = int(config.get("reference_cap_per_trial", 128))
    for held in TRAIN_REPS:
        train_reps = tuple(rep for rep in TRAIN_REPS if rep != held)
        fold = root / "oof" / f"held_rep_{held}"
        fold.mkdir(parents=True, exist_ok=True)
        subset = fold_inventory(inventory, train_reps, [held])
        mean, std, provenance = fit_input_scaler(segments, subset, train_reps)
        atomic_npz(fold / "input_scaler.npz", mean=mean, std=std)
        json_write(fold / "input_scaler_provenance.json", provenance)
        train = make_windows(segments, subset, mean, std, "train", **window_options)
        evaluation = make_windows(segments, subset, mean, std, "evaluation", **window_options)
        train.meta.to_csv(fold / "train_windows.csv", index=False)
        evaluation.meta.to_csv(fold / "held_windows.csv", index=False)
        train_indices = metadata_indices(development.meta, train.meta)
        held_indices = metadata_indices(development.meta, evaluation.meta)
        reference, selected = fit_shift_reference(development_raw[train_indices], train.meta, train_reps, cap)
        json_write(fold / "shift_reference.json", reference)
        train.meta.iloc[selected].to_csv(fold / "reference_windows.csv", index=False)
        oof_shifts[held_indices] = apply_shift_reference(development_raw[held_indices], reference)
        for arm_index, arm in enumerate(EXPERTS):
            model = train_fit(arm, seed, subject, train, fold / arm, f"oof_held_{held}", run_signature, config)
            logits = model_predict(model, evaluation, int(config.get("batch_size", 512)))
            save_predictions(fold / arm / "held", evaluation, logits)
            oof_logits[held_indices, arm_index] = logits
            del model
            gc.collect()
            torch.cuda.empty_cache()
        covered[held_indices] += 1
        fold_records.append(dict(held_repetition=held, training_repetitions=list(train_reps),
                                 training_window_sha256=window_hash(train.meta),
                                 held_window_sha256=window_hash(evaluation.meta),
                                 input_scaler_sha256=file_hash(fold / "input_scaler.npz"),
                                 shift_reference_sha256=reference["sha256"]))
        del train, evaluation
    if not np.all(covered == 1) or not np.isfinite(oof_logits).all() or not np.isfinite(oof_shifts).all():
        raise RuntimeError("OOF predictions do not cover every development window exactly once")
    oof_meta = development.meta.assign(oof_held_repetition=development.meta.native_repetition)
    oof_meta.to_csv(root / "oof_metadata.csv", index=False)
    atomic_npz(root / "oof_bundle.npz", logits=oof_logits, y=development.y,
               repetitions=development.meta.native_repetition.to_numpy(), shifts=oof_shifts,
               physical_features=development_raw)
    json_write(root / "oof_provenance.json", dict(arm_order=list(EXPERTS), folds=fold_records,
                                                test_repetitions_used=False,
                                                development_window_sha256=window_hash(development.meta)))
    # All calibration/gating/BRB fitting occurs here, before even building test windows.
    meta_folder = root / "meta"
    meta_folder.mkdir(exist_ok=True)
    meta_model = brb_meta.fit_meta(oof_logits, development.y,
                                   development.meta.native_repetition.to_numpy(), oof_shifts,
                                   meta_folder, metadata=dict(subject=subject, seed=seed,
                                   arm_order=list(EXPERTS), run_signature=run_signature,
                                   fold_provenance=fold_records, smoke=smoke,
                                   window_metadata_path="../oof_metadata.csv",
                                   window_metadata_sha256=file_hash(root / "oof_metadata.csv"),
                                   development_windows=len(oof_meta)))
    json_write(meta_folder / "locked_meta_model.json", meta_model)
    locked_meta_hash = file_hash(meta_folder / "locked_meta_model.json")
    json_write(meta_folder / "LOCKED_BEFORE_TEST.json", dict(
        model_sha256=locked_meta_hash, test_predictions_available=False,
        fit_inputs="training-only neural OOF logits/labels/repetitions and fold-fitted shifts",
        test_usage="secondary descriptive evaluation; hypotheses already informed by previous test results"))
    final_folder = root / "final"
    final_folder.mkdir(exist_ok=True)
    atomic_npz(final_folder / "input_scaler.npz", mean=final_mean, std=final_std)
    json_write(final_folder / "input_scaler_provenance.json", final_provenance)
    final_reference, selected = fit_shift_reference(development_raw, development.meta, TRAIN_REPS, cap)
    json_write(final_folder / "shift_reference.json", final_reference)
    development.meta.iloc[selected].to_csv(final_folder / "reference_windows.csv", index=False)
    for arm in FINAL_ARMS:
        model = train_fit(arm, seed, subject, development, final_folder / arm, "final", run_signature, config)
        del model
        gc.collect()
        torch.cuda.empty_cache()
    checkpoint_hashes = {arm: file_hash(final_folder / arm / "final_model.pt") for arm in FINAL_ARMS}
    json_write(root / "ALL_MODELS_LOCKED_BEFORE_TEST.json", dict(meta_model_sha256=locked_meta_hash,
                                                               checkpoint_sha256=checkpoint_hashes,
                                                               final_shift_reference_sha256=final_reference["sha256"]))
    test = make_windows(segments, complete_inventory, final_mean, final_std, "evaluation", **window_options)
    test.meta["seed"] = int(seed)
    if set(test.meta.native_repetition) != set(TEST_REPS):
        raise RuntimeError("Outer test split must contain only repetitions 2/5")
    test.meta.to_csv(root / "test_metadata.csv", index=False)
    test_raw = dataset_descriptors(test)
    test_shifts = apply_shift_reference(test_raw, final_reference)
    test_logits = np.empty((len(test), 3, 17), dtype=np.float32)
    controls = {}
    for arm in FINAL_ARMS:
        model = AblationC1(arm, config.get("dropout", 0.65)).cuda()
        model.load_state_dict(torch.load(final_folder / arm / "final_model.pt", map_location="cuda", weights_only=True))
        logits = model_predict(model, test, int(config.get("batch_size", 512)))
        probs = save_predictions(final_folder / arm / "test", test, logits)
        if arm in EXPERTS:
            test_logits[:, EXPERTS.index(arm)] = logits
        else:
            controls[arm] = probs
        del model
        gc.collect()
        torch.cuda.empty_cache()
    if file_hash(meta_folder / "locked_meta_model.json") != locked_meta_hash:
        raise RuntimeError("Meta model changed after locking")
    predictions = brb_meta.predict_meta(meta_model, test_logits, test_shifts)
    diagnostics = brb_meta.predict_diagnostics(meta_model, test_logits, test_shifts)
    results = export_meta_results(root / "results", test.meta, test.y, predictions, diagnostics,
                                  controls["SI"], controls["WSI"])
    json_write(root / "meta_completion.json", dict(subject=subject, seed=seed, success=True,
                                                  meta_model_sha256=locked_meta_hash,
                                                  methods=[row["method"] for row in results],
                                                  smoke=smoke))
    atomic_npz(root / "subject_bundle.npz", oof_logits=oof_logits, oof_y=development.y,
               oof_repetitions=development.meta.native_repetition.to_numpy(), oof_shifts=oof_shifts,
               test_logits=test_logits, test_y=test.y, test_shifts=test_shifts,
               test_physical_features=test_raw, control_si=controls["SI"], control_wsi=controls["WSI"])
    complete = dict(success=True, subject=subject, seed=seed, neural_fits=17,
                    oof_fits=12, final_fits=5, arm_order=list(EXPERTS),
                    run_signature=run_signature, meta_model_sha256=locked_meta_hash,
                    test_window_sha256=window_hash(test.meta), test_windows=len(test),
                    smoke=smoke, smoke_label="NONRESEARCH SMOKE" if smoke else None,
                    methods=[row["method"] for row in results])
    json_write(root / "SUBJECT_COMPLETE.json", complete)
    print(f"SUBJECT_COMPLETE S{subject:02} seed{seed} neural_fits=17", flush=True)
    del development, test, segments
    gc.collect()
    torch.cuda.empty_cache()
    return complete


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", type=Path, required=True)
    parser.add_argument("--gpu-id", type=int, required=True)
    parser.add_argument("--subjects", required=True, help="comma-separated subject IDs assigned to this GPU")
    parser.add_argument("--output", type=Path, required=True)
    args = parser.parse_args(argv)
    args.output.mkdir(parents=True, exist_ok=True)
    try:
        config = json.loads(args.config.read_text(encoding="utf-8-sig"))
        validate_config(config)
        subjects = [int(value) for value in args.subjects.split(",")]
        if len(set(subjects)) != len(subjects) or not set(subjects).issubset(config["subjects"]):
            raise ValueError("Worker subjects must be a unique subset of configured subjects")
        # The launcher sets CUDA_VISIBLE_DEVICES before this process imports torch.
        import torch
        import tc_support as support
        torch.set_num_threads(int(config.get("cpu_threads_per_worker", 2)))
        support.Settings.TRAIN_REPS = list(TRAIN_REPS)
        support.Settings.TEST_REPS = list(TEST_REPS)
        support.Settings.FS, support.Settings.WINDOW, support.Settings.STEP = 2000, 400, 20
        support.Settings.BATCH_SIZE, support.Settings.DROPOUT = 512, 0.65
        scripts = Path(__file__).resolve().parent
        code_hashes = {path.name: file_hash(path) for path in sorted(scripts.glob("*.py"))}
        signature = digest(dict(config=config, code=code_hashes))
        run_record = args.output / f"worker_{args.gpu_id}_configuration.json"
        if run_record.exists():
            existing = json.loads(run_record.read_text(encoding="utf-8"))
            if existing["run_signature"] != signature:
                raise RuntimeError("Output directory belongs to a different code/configuration")
        json_write(run_record, dict(config=config, code_sha256=code_hashes, run_signature=signature,
                                    assigned_subjects=subjects, physical_gpu=args.gpu_id))
        gpu_preflight(args.gpu_id, args.output)
        completed = []
        for subject in subjects:
            for seed in config["seeds"]:
                completed.append(run_subject(subject, int(seed), config, args.output, signature))
        json_write(args.output / f"worker_{args.gpu_id}_complete.json",
                   dict(success=True, physical_gpu=args.gpu_id, subjects=subjects,
                        neural_fits=sum(record["neural_fits"] for record in completed),
                        run_signature=signature, completions=completed))
    except Exception:
        (args.output / f"gpu{args.gpu_id}_failure.txt").write_text(traceback.format_exc(), encoding="utf-8")
        raise


if __name__ == "__main__":
    main()


### brb_launch.py

Two-GPU coordination, complete smoke test, fit-coverage checks and result archive.


In [ ]:
%%writefile /kaggle/working/db7_brb_src/brb_launch.py
"""Kaggle coordinator: two isolated GPU workers, smoke checks, verified archive.

The research scope is fixed at S1/S15, seed42. Smoke outputs are separate and
never counted as research results. Failure still produces a partial archive.
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import threading
import time
import traceback
import zipfile
from datetime import datetime, timezone


def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2), encoding="utf-8")


def expected_fit_keys(subjects, seeds):
    return {(s, seed, arm, stage, held) for s in subjects for seed in seeds
            for stage, arms, helds in [("oof", ("W", "S", "I"), (1,3,4,6)),
                                      ("final", ("W", "S", "I", "SI", "WSI"), (None,))]
            for arm in arms for held in helds}


def validate_fits(folder, config, epochs):
    expected = expected_fit_keys(config["subjects"], config["seeds"])
    actual = set()
    files = list(Path(folder).rglob("fit_manifest.json"))
    for path in files:
        row = json.loads(path.read_text())
        key = (row["subject"], row["seed"], row["arm"], row["stage"],
               row.get("held_out_repetition"))
        if row.get("status") != "complete" or row.get("epochs_run") != epochs:
            raise RuntimeError(f"Incomplete fit: {path}")
        if key in actual:
            raise RuntimeError(f"Duplicate fit: {key}")
        actual.add(key)
    if actual != expected:
        raise RuntimeError(f"Fit coverage mismatch: missing={expected-actual}, extra={actual-expected}")
    meta = [json.loads(p.read_text()) for p in Path(folder).rglob("meta_completion.json")]
    meta_keys = [(x["subject"], x["seed"]) for x in meta if x.get("success")]
    if sorted(meta_keys) != sorted((s,k) for s in config["subjects"] for k in config["seeds"]):
        raise RuntimeError("Meta completion coverage mismatch")
    return len(files), len(meta)


def telemetry(folder):
    result = subprocess.run(["nvidia-smi", "--query-gpu=index,uuid,name,utilization.gpu,memory.used",
                             "--format=csv,noheader,nounits"], capture_output=True, text=True)
    line = {"utc": datetime.now(timezone.utc).isoformat(), "returncode": result.returncode,
            "gpus": result.stdout.strip().splitlines()}
    with (folder / "gpu_telemetry.jsonl").open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(line)+"\n")
    print("GPU telemetry", line, flush=True)


def run_workers(config, folder):
    """Child visibility is set before torch import; assign by position, not parity."""
    folder.mkdir(parents=True, exist_ok=True)
    config_path = folder / "config.json"
    write_json(config_path, config)
    processes = []
    threads = []
    start = time.monotonic()

    def pump(pipe, path, label):
        with path.open("w", encoding="utf-8") as handle:
            for line in pipe:
                handle.write(line)
                handle.flush()
                print(f"[{label}] {line.rstrip()}", flush=True)

    try:
        for gpu, subject in enumerate(config["subjects"]):
            env = os.environ.copy()
            env.update(CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1",
                       OMP_NUM_THREADS="2", MKL_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2")
            args = [sys.executable, str(Path(__file__).with_name("brb_worker.py")),
                    "--config", str(config_path), "--gpu-id", str(gpu),
                    "--subjects", str(subject), "--output", str(folder / f"S{subject:02d}")]
            process = subprocess.Popen(args, env=env, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, bufsize=1)
            processes.append(process)
            thread = threading.Thread(target=pump, args=(process.stdout,
                    folder/f"gpu{gpu}_worker.log", f"GPU{gpu}/S{subject}"), daemon=True)
            thread.start()
            threads.append(thread)
        last = 0
        while any(p.poll() is None for p in processes):
            if any(p.poll() not in (None, 0) for p in processes):
                raise RuntimeError("A GPU worker failed; stopping paired worker")
            if time.monotonic()-start > (900 if config["smoke"] else 16200):
                raise TimeoutError("Coordinator time limit reached; saving partial results")
            if time.monotonic()-last >= 30:
                telemetry(folder)
                last = time.monotonic()
            time.sleep(2)
        if any(p.returncode != 0 for p in processes):
            raise RuntimeError(f"Worker exits: {[p.returncode for p in processes]}")
    finally:
        for process in processes:
            if process.poll() is None:
                process.terminate()
        for process in processes:
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
        for thread in threads:
            thread.join(timeout=5)
    return validate_fits(folder, config, config["epochs"])


def aggregate(folder):
    """Combine the per-subject report tables without treating windows as IID."""
    import pandas as pd
    paths = sorted((folder / "full").rglob("method_metrics.csv"))
    if not paths:
        raise RuntimeError("Missing per-subject method_metrics.csv")
    frames = [pd.read_csv(p) for p in paths]
    table = pd.concat(frames, ignore_index=True)
    table.to_csv(folder / "all_subject_metrics.csv", index=False)
    text = ["# DB7-017 exploratory pilot", "", "S1 and S15, seed42; 34 verified neural fits.",
            "", "Test repetitions2/5 were previously inspected. This pilot checks feasibility and paired recovery; it is not a confirmatory population result.",
            "", "Per-subject method metrics:", "", "```", table.to_string(index=False), "```", "",
            "Use subject test outputs for recovery/harm, rule support, reliability calibration, phase and gesture diagnostics. Smoke files are implementation checks only."]
    (folder / "RESULTS.md").write_text("\n".join(text), encoding="utf-8")


def main(config_path):
    import torch
    config = json.loads(Path(config_path).read_text())
    if config["subjects"] != [1,15] or config["seeds"] != [42] or config["epochs"] != 13:
        raise ValueError("Only the authorized two-subject, seed42, 13-epoch pilot is supported")
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    folder = Path("/kaggle/working") / f"db7_brb_pilot_{stamp}"
    folder.mkdir(parents=True)
    scripts = folder / "source"
    scripts.mkdir()
    hashes = {}
    for source in Path(__file__).parent.glob("*.py"):
        shutil.copy2(source, scripts / source.name)
        hashes[source.name] = hashlib.sha256(source.read_bytes()).hexdigest()
    write_json(folder / "source_sha256.json", hashes)
    if config.get("source_sha256") != hashes:
        raise RuntimeError("Packaged source hashes do not match the executed modules")
    write_json(folder / "run_manifest.json", dict(config, source_sha256=hashes))
    write_json(folder / "environment.json", {"python":sys.version,"torch":torch.__version__,
                                             "cuda":torch.version.cuda})
    preflight = {"success":False,"gpu_count":torch.cuda.device_count(),"smoke_success":False}
    completion = {"experiment_id":"DB7-017","success":False,"subjects":[1,15],"seeds":[42],
                  "neural_fits":0,"meta_completed":0}
    error = None
    try:
        if preflight["gpu_count"] != 2:
            raise RuntimeError(f"Two GPUs required; found {preflight['gpu_count']}")
        preflight["gpus"] = [torch.cuda.get_device_name(i) for i in range(2)]
        print("DB7-017 TWO-GPU PREFLIGHT", preflight, flush=True)
        smoke = dict(config, smoke=True, epochs=2)
        run_workers(smoke, folder / "smoke")
        preflight.update(success=True, smoke_success=True)
        write_json(folder / "preflight.json", preflight)
        print("DB7-017 TWO-GPU SMOKE PASSED; STARTING 34 RESEARCH FITS", flush=True)
        fits, metas = run_workers(config, folder / "full")
        aggregate(folder)
        completion.update(success=True, neural_fits=fits, meta_completed=metas)
    except Exception as exc:
        error = exc
        completion["error"] = repr(exc)
        completion["traceback"] = traceback.format_exc()
        print(completion["traceback"], flush=True)
    finally:
        write_json(folder / "preflight.json", preflight)
        write_json(folder / "completion.json", completion)
        archive = folder.with_suffix(".zip")
        with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED, compresslevel=3) as zf:
            for path in sorted(folder.rglob("*")):
                if path.is_file():
                    zf.write(path, path.relative_to(folder).as_posix())
        digest = hashlib.sha256()
        with archive.open("rb") as handle:
            for block in iter(lambda: handle.read(1024*1024), b""):
                digest.update(block)
        print("RESULT_ARCHIVE", archive, "SHA256", digest.hexdigest(), flush=True)
    if error is not None:
        raise error
    print("DB7-017 COMPLETE", completion, flush=True)
    return str(folder)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    main(parser.parse_args().config)


## Execute the approved pilot

The coordinator requires two GPUs, runs a small complete smoke pipeline on each, then starts the 34 research fits. A worker failure stops its paired worker and saves partial output. GitHub collects the archive and verifies every full fit and meta completion.


In [ ]:
# Frozen, human-readable experiment configuration.
CONFIG = json.loads(r'''
{
  "experiment_id": "DB7-017",
  "name": "BRB reliability fusion pilot",
  "subjects": [
    1,
    15
  ],
  "seeds": [
    42
  ],
  "smoke": false,
  "epochs": 13,
  "batch_size": 512,
  "dropout": 0.65,
  "window_samples": 400,
  "stride_samples": 20,
  "fs": 2000,
  "train_repetitions": [
    1,
    3,
    4,
    6
  ],
  "test_repetitions": [
    2,
    5
  ],
  "reference_cap_per_trial": 128,
  "smoke_windows_per_trial": 32,
  "smoke_max_train_batches": 2,
  "expected_neural_fits": 34,
  "exercise": "B: E1 labels 1-17; rest excluded",
  "channels": {
    "emg": 12,
    "acc": 36,
    "gyro": 36,
    "mag": 36
  },
  "emg_filter": "Per annotated segment: 4th-order 20-450 Hz zero-phase bandpass, 50 Hz Q30 zero-phase notch",
  "inertial": "Segment-local resample_poly to EMG grid; preserve offsets; no per-window centering",
  "normalization": "Training samples only; spectral moments on training windows only",
  "optimizer": "Adam; lr 0.001 epochs1-3, 0.0001 epochs4-9, 0.00001 epochs10-13; weight_decay0; clip5",
  "augmentation": false,
  "oof": "W/S/I each rotate held-out repetition among1/3/4/6; train on other3;12 fits per subject",
  "final": "W/S/I/SI/WSI each train on all4 training repetitions;5 fits per subject",
  "meta_fit_repetitions": [
    1,
    3,
    4
  ],
  "reliability_calibration_repetition": 6,
  "meta_validation_note": "OOF experts share underlying training recordings across meta groups. Rep6 is a development calibration holdout, not a fully nested independent validation estimate. Hyperparameters are fixed; no test selection.",
  "evaluation_note": "Exploratory pilot. Test repetitions2/5 were examined in previous studies; no claim of fresh confirmatory evaluation or population significance.",
  "brb": "Three8-rule binary-consequent BRBs; fixed low/high references and equal premise weights; train24 consequent logits; analytical RIMER within each BRB; calibrated reliability times frozen global importance for probability averaging",
  "cross_expert_er": false,
  "gpu_assignment": {
    "0": [
      1
    ],
    "1": [
      15
    ]
  },
  "source_sha256": {
    "three_branch_model.py": "d3cc70879dd84abead54cde20da12fe5ef772f572c346c1fa45f550d639c4c7d",
    "ablation_model.py": "e81fefd8f27ffd9de5a2f7c5130bd1bd2e7a5e31671d047b53eb7bf9a370cbbd",
    "tc_support.py": "029c19915166759a5605d6a5a236290bce7599928df3e8b7fdbf48aa3d9dda17",
    "brb_meta.py": "22d1b57d0f2f8eecbd0abe02c2667d277bdb531b48993a7d77ce5d301a173854",
    "brb_worker.py": "a1235da080108a5e71572037cca6d79ae08b02f60c1105a35713101ddf4bfb16",
    "brb_launch.py": "d65093ad8bcb5912f517b4e1f7ec27792a855583573bd972330c806569328ab3"
  },
  "protocol_sha256": "7f20b0e28797cb4a57b48c9dd086812fbbbbd1d517cd3903a084b8b9455c6a20"
}
''')
config_path = SOURCE / 'protocol.json'
config_path.write_text(json.dumps(CONFIG, indent=2))
print(json.dumps(CONFIG, indent=2))


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, str(SOURCE / 'brb_launch.py'),
                '--config', str(config_path)], check=True)


## Outputs and decision criteria

Start with `RESULTS.md` and `all_subject_metrics.csv` inside `db7_brb_pilot_*.zip`.

For each subject the archive contains frozen models/scalers, OOF and test logits, rule beliefs/support, calibration records, per-window expert probabilities, reliability indicators/weights, per-gesture/repetition/phase errors, confusion matrices and recovery/harm against SI and WSI. Histories and GPU telemetry document training and device use. Smoke results live separately under `smoke/` and never contribute to the 34-fit counts or research metrics.

Compare BRB with global weights, confidence weighting, logistic reliability and Sugeno fuzzy reliability on exactly the same predictions. Compare BRB without shift to test whether the physical-signal deviation indicator helps. Check net recovery (corrected minus newly introduced errors), calibration and rule support. Improvement over weaker standalone experts alone is insufficient; SI and WSI are the primary controls. Cross-expert ER is not part of this pilot; RIMER is used internally to combine each BRB's rules.
